# 캐릭터 LoRA 학습 노트북 (diffusers + peft)

이 노트북은 이미지 생성 모델에 캐릭터 LoRA 를 학습한다. 베이스 모델은 얼리고 LoRA 어댑터만 학습한다.

| 단계 | 하는 일 |
|---|---|
| 0. 준비 | 라이브러리·버전 확인, 설정(고정값), 모델 계열별 어댑터, 모델 불러오기 |
| 1. EDA | 데이터셋 이미지·캡션을 읽고 크기·비율·중복·흐림·트리거를 검사 |
| 2. 전처리 | 비율별 묶음(bucket)으로 자르고, 이미지 잠재값과 텍스트 임베딩을 미리 계산 |
| 3. 시각화 | 전처리 결과, 비율·캡션 분포, 잠재값 통계를 그림으로 확인 |
| 4. 학습 | LoRA 를 붙여 학습하고, 정해진 단계마다 저장·검증 |
| 5. 검증 | 손실 곡선, 단계별 샘플 비교, 저장 파일 다시 불러오기 확인, 결과 요약 |

- 지원 모델 계열: `qwen_image`(Qwen-Image, 2512), `qwen_image21`, `flux2_klein`, `z_image`, `krea2`, `flux1`, `chroma`, `longcat_image`, `sdxl`, `sd3`.
- 사용자가 바꾸는 값은 `0-2` 의 `USER` 뿐이다. 학습 조건(랭크, 학습률, optimizer, 단계 수 등)은 `FIXED` 에 고정되어 있고, 실행 중에 바꿀 수 없다.
- 모델은 diffusers 형식 폴더(`model_index.json` 이 있는 폴더) 또는 Hugging Face 저장소 이름(예: `Qwen/Qwen-Image-2512`)으로 불러온다. 폴더면 `model_index.json` 의 파이프라인 종류가 `MODEL_FAMILY` 와 맞는지 먼저 확인한다.
- 메인 서버 ComfyUI 의 단일 `.safetensors` 파일(예: `qwen_image_2512_fp8_e4m3fn.safetensors`)은 그대로 쓸 수 없다. 같은 모델의 diffusers 형식 폴더를 따로 받아 둔다.
- 노트북은 실행한 폴더에 `lora_adapters/` 를 만든다.

### 데이터셋 폴더

```
dataset/nkgreen/
  nkgreen_01.png   nkgreen_01.txt   # 같은 이름의 .txt 에 영문 캡션 한 줄
  nkgreen_02.png   nkgreen_02.txt
  ...
```

- 캡션은 `nkgreen, full body, front view, standing, plain light gray background, clean anime illustration` 처럼 트리거 + 바뀌는 것만 적는다 (`prompts/nkgreen_qwen2512_prompts_*.md` 참고).
- 캡션 파일이 없으면 트리거만으로 학습하고 경고를 남긴다.

### 시험 실행 (오류 검사)

환경변수 `LORA_NB_TEST=<모델 계열>` 을 주고 실행하면, 같은 구조의 아주 작은 무작위 모델과 가짜 데이터셋으로 모든 셀을 끝까지 돌린다 (계열당 CPU 30초 안팎). 실제 학습 전에 환경이 맞는지 확인할 때 쓴다. 작은 토크나이저를 Hugging Face 에서 내려받으므로 인터넷이 필요하다. `LORA_NB_TEST_DEVICE=cuda` 를 더하면 GPU·bf16 으로 시험한다.

```
LORA_NB_TEST=qwen_image jupyter nbconvert --to notebook --execute lora_training.ipynb --output test_run.ipynb
```

## 0. 준비

### 0-1. 라이브러리 버전 확인과 불러오기

In [ ]:
# 고정 버전: 이 버전 조합에서 모든 모델 계열을 작은 무작위 모델로 시험했다.
#   pip install torch==2.15.0 diffusers==0.42.1 peft==0.21.2 transformers==5.22.1 accelerate==1.15.0 \
#               safetensors sentencepiece protobuf pillow numpy pandas matplotlib tqdm
import importlib.metadata as _md

PINNED = {"torch": "2.15.0", "diffusers": "0.42.1", "peft": "0.21.2", "transformers": "5.22.1", "accelerate": "1.15.0"}
_rows = []
for _pkg, _want in PINNED.items():
    try:
        _have = _md.version(_pkg)
    except _md.PackageNotFoundError:
        raise ImportError(f"{_pkg} 가 설치되어 있지 않습니다. 위 pip 명령으로 {_pkg}=={_want} 를 설치하세요.")
    _ok = _have.split("+")[0] == _want
    _rows.append((_pkg, _want, _have, "OK" if _ok else "버전 다름"))
for _r in _rows:
    print(f"{_r[0]:<13} 고정 {_r[1]:<8} 설치 {_r[2]:<14} {_r[3]}")
if any(r[3] != "OK" for r in _rows):
    print("\n경고: 고정 버전과 다릅니다. 시험하지 않은 조합이라 계열에 따라 오류가 날 수 있습니다.")

import contextlib, copy, gc, hashlib, importlib, json, math, os, random, re, shutil, sys, tempfile, time, types
from collections import Counter
from pathlib import Path
from types import MappingProxyType

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

import torch
import diffusers
from diffusers.optimization import get_scheduler
from peft import LoraConfig, get_peft_model_state_dict, set_peft_model_state_dict
from tqdm.auto import tqdm

if not hasattr(sys, "ps1") and "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

### 0-2. 설정

- `USER` 만 바꾼다. 환경변수로도 줄 수 있다 (예: `LORA_MODEL_FAMILY`, `LORA_MODEL_PATH`).
- `FIXED` 는 계획서(`lora-training-plan.md` 3-3 절)와 diffusers 공식 학습 스크립트 기본값을 따른 고정값이다. 읽기 전용이라 실행 중에 바꾸면 오류가 난다.

In [ ]:
TEST_FAMILY = os.environ.get("LORA_NB_TEST", "").strip()   # 시험 실행이면 모델 계열 이름

USER = MappingProxyType(dict(
    MODEL_FAMILY=os.environ.get("LORA_MODEL_FAMILY", TEST_FAMILY or "qwen_image"),
    MODEL_PATH=os.environ.get("LORA_MODEL_PATH", "/srv/models/diffusers/Qwen-Image-2512"),
    DATASET_DIR=os.environ.get("LORA_DATASET_DIR", "dataset/nkgreen"),
    OUTPUT_DIR=os.environ.get("LORA_OUTPUT_DIR", "outputs"),
    TRIGGER=os.environ.get("LORA_TRIGGER", "nkgreen"),
    RUN_NAME=os.environ.get("LORA_RUN_NAME", "nkgreen_v1"),
))

FIXED = MappingProxyType(dict(
    seed=42,
    # LoRA
    rank=16, alpha=16, dropout=0.0, init_lora_weights="gaussian",
    # optimizer (torch.optim.AdamW)
    optimizer="AdamW", learning_rate=1e-4, betas=(0.9, 0.999), weight_decay=1e-4, eps=1e-8,
    lr_scheduler="constant_with_warmup", warmup_steps=100,
    # 학습 길이
    max_train_steps=1500, batch_size=1, grad_accum_steps=1, max_grad_norm=1.0,
    # 해상도: 한 변 1024 의 넓이를 기준으로 비율별 묶음
    resolution=1024, bucket_aspects=(0.5, 0.5625, 0.625, 0.6667, 0.75, 0.8, 1.0, 1.25, 1.3333, 1.5, 1.6, 1.7778, 2.0),
    # 노이즈 단계 뽑기: flow 계열은 logit-normal(0,1), eps/v 계열은 균등
    timestep_sampling="logit_normal", logit_mean=0.0, logit_std=1.0,
    # 정밀도·메모리
    mixed_precision="bf16", gradient_checkpointing=True,
    # 저장·검증
    save_every=250, validate_every=250, val_loss_images=4, val_sigmas=(0.1, 0.3, 0.5, 0.7, 0.9),
    max_nan_steps=10, min_image_side=512,
))

TEST_DEVICE = os.environ.get("LORA_NB_TEST_DEVICE", "cpu")   # 시험 실행 장치: cpu(fp32) 또는 cuda(bf16)
if TEST_FAMILY:   # 시험 실행: 같은 코드 경로를 아주 작은 값으로
    FIXED = MappingProxyType({**FIXED, "max_train_steps": 4, "warmup_steps": 1, "save_every": 2, "validate_every": 2,
                              "val_loss_images": 2, "min_image_side": 8,
                              "mixed_precision": "bf16" if TEST_DEVICE == "cuda" else "fp32"})

VALIDATION_PROMPTS = [
    f"{USER['TRIGGER']}, full body, front view, standing, plain light gray background, clean anime illustration",
    f"{USER['TRIGGER']}, upper body, three-quarter view from the left, shy small smile, summer meadow, clean anime illustration",
    f"{USER['TRIGGER']}, close-up, side view, calm look into the distance, plain light gray background, clean anime illustration",
    f"{USER['TRIGGER']}, full body, back view, standing, forest path, clean anime illustration",
]
VALIDATION_NEGATIVE = "photorealistic, 3d render, text, watermark, blurry, deformed, extra limbs, hands, fingers, multiple people"
VALIDATION_SEEDS = [1001, 1002, 1003, 1004]
print(json.dumps(dict(USER), ensure_ascii=False, indent=1))

### 0-3. 모델 계열별 어댑터 (수정하지 않는다)

모델마다 텍스트 인코딩, 잠재값 정규화, 노이즈 공식, 잠재값 묶기(packing), LoRA 대상 층, 저장 형식이 다르다.
아래 셀들은 계열마다 같은 함수 이름(`encode_prompts`, `encode_images`, `forward_loss`, `save_lora`, `generate`)을 가진 파일을 `lora_adapters/` 에 쓰고, 선택한 계열 하나만 불러온다.
각 어댑터는 diffusers v0.42.1 의 공식 학습 스크립트와 대조했다 (diffusers 에 스크립트가 없는 longcat_image 는 Meituan 공식 LoRA 스크립트). 모든 계열은 아주 작은 무작위 모델로 CPU(fp32)와 GPU(bf16)에서 처음부터 끝까지 실행해 확인했고, 실제 가중치로는 아직 학습해 보지 않았다.

In [ ]:
os.makedirs("lora_adapters", exist_ok=True)

In [ ]:
%%writefile lora_adapters/qwen_image.py
"""Qwen-Image adapter (QwenImagePipeline / QwenImageTransformer2DModel).

Mirrors diffusers v0.42.1 examples/dreambooth/train_dreambooth_lora_qwen_image.py:
  * latents: vae.encode(pixels[:, :, None]).latent_dist.sample(); x0 = (z - latents_mean) / latents_std  (5D B,C,1,h,w)
  * flow matching: x_t = (1 - s) * x0 + s * noise ; target = noise - x0
  * transformer timestep = timesteps / 1000 = sigma (0..1)
  * latents permuted to (B,1,C,h,w) then QwenImagePipeline._pack_latents (2x2 patches), img_shapes=[(1,h/2,w/2)]
  * encoder_hidden_states_mask passed (None when all ones, as encode_prompt does)
  * LoRA targets: ["to_k", "to_q", "to_v", "to_out.0"]
"""
import os

import torch
from diffusers import (
    AutoencoderKLQwenImage,
    FlowMatchEulerDiscreteScheduler,
    QwenImagePipeline,
    QwenImageTransformer2DModel,
)

FAMILY = "qwen_image"
PIPELINE_CLASS = QwenImagePipeline
DENOISER_ATTR = "transformer"
PREDICTION = "flow"
PIXEL_MULTIPLE = 16  # vae_scale_factor (8) * 2x2 packing
DEFAULT_STEPS = 50  # QwenImagePipeline.__call__ default num_inference_steps
DEFAULT_GUIDANCE = 4.0  # true_cfg_scale default (Qwen-Image uses true CFG, no guidance embedding)
LORA_TARGET_MODULES = ["to_k", "to_q", "to_v", "to_out.0"]
MAX_SEQUENCE_LENGTH = 512  # official script --max_sequence_length default


def load_pipeline(model_path: str, dtype: torch.dtype):
    # Works for a local diffusers-format dir (model_index.json) or a HF repo id.
    return QwenImagePipeline.from_pretrained(model_path, torch_dtype=dtype)


def tiny_pipeline(dtype: torch.dtype):
    # Same as diffusers tests/pipelines/qwenimage/test_qwenimage.py get_dummy_components()
    from transformers import Qwen2_5_VLConfig, Qwen2_5_VLForConditionalGeneration, Qwen2Tokenizer

    torch.manual_seed(0)
    transformer = QwenImageTransformer2DModel(
        patch_size=2,
        in_channels=16,
        out_channels=4,
        num_layers=2,
        attention_head_dim=16,
        num_attention_heads=3,
        joint_attention_dim=16,
        guidance_embeds=False,
        axes_dims_rope=(8, 4, 4),
    )
    torch.manual_seed(0)
    z_dim = 4
    vae = AutoencoderKLQwenImage(
        base_dim=z_dim * 6,
        z_dim=z_dim,
        dim_mult=[1, 2, 4],
        num_res_blocks=1,
        temperal_downsample=[False, True],
        latents_mean=[0.0] * 4,
        latents_std=[1.0] * 4,
    )
    torch.manual_seed(0)
    scheduler = FlowMatchEulerDiscreteScheduler()
    torch.manual_seed(0)
    config = Qwen2_5_VLConfig(
        text_config={
            "hidden_size": 16,
            "intermediate_size": 16,
            "num_hidden_layers": 2,
            "num_attention_heads": 2,
            "num_key_value_heads": 2,
            "rope_scaling": {"mrope_section": [1, 1, 2], "rope_type": "default", "type": "default"},
            "rope_theta": 1000000.0,
        },
        vision_config={"depth": 2, "hidden_size": 16, "intermediate_size": 16, "num_heads": 2, "out_hidden_size": 16},
        hidden_size=16,
        vocab_size=152064,
        vision_end_token_id=151653,
        vision_start_token_id=151652,
        vision_token_id=151654,
    )
    text_encoder = Qwen2_5_VLForConditionalGeneration(config).eval()
    tokenizer = Qwen2Tokenizer.from_pretrained("hf-internal-testing/tiny-random-Qwen2VLForConditionalGeneration")
    pipe = QwenImagePipeline(
        scheduler=scheduler, vae=vae, text_encoder=text_encoder, tokenizer=tokenizer, transformer=transformer
    )
    return pipe.to(dtype=dtype)


@torch.no_grad()
def encode_prompts(pipe, prompts, device, dtype):
    out = []
    for p in prompts:  # one at a time -> no cross-prompt padding
        emb, mask = pipe.encode_prompt(prompt=[p], device=device, max_sequence_length=MAX_SEQUENCE_LENGTH)
        if mask is None:
            mask = torch.ones(emb.shape[:2], dtype=torch.long, device=emb.device)
        out.append({
            "prompt_embeds": emb.to(dtype=dtype).cpu(),
            "prompt_embeds_mask": mask.to(torch.long).cpu(),
        })
    return out


def _latent_stats(vae, device, dtype):
    z = vae.config.z_dim
    mean = torch.tensor(vae.config.latents_mean).view(1, z, 1, 1, 1).to(device, dtype)
    std = torch.tensor(vae.config.latents_std).view(1, z, 1, 1, 1).to(device, dtype)
    return mean, std


@torch.no_grad()
def encode_images(pipe, pixel_values, device, dtype):
    vae = pipe.vae
    px = pixel_values.to(device=device, dtype=vae.dtype)
    if px.ndim == 4:
        px = px.unsqueeze(2)  # (B,3,1,H,W): Qwen VAE is a video VAE
    lat = vae.encode(px).latent_dist.sample()  # official script samples
    mean, std = _latent_stats(vae, lat.device, lat.dtype)
    lat = (lat - mean) / std
    return lat.float().cpu()  # (B, C, 1, h, w)


def _mask_or_none(mask):
    if mask is None or bool(mask.all()):
        return None
    return mask


def collate_conds(conds):
    """Batch a list of per-prompt cond dicts (each batch 1) into one cond for forward_loss.

    Mirrors concat_prompt_embedding_batches in the official train_dreambooth_lora_qwen_image.py:
    embeds are zero-padded to the longest sequence and padded positions get mask 0.
    """
    if not conds:
        raise ValueError("collate_conds needs at least one cond")
    target = max(c["prompt_embeds"].shape[1] for c in conds)
    embs, masks = [], []
    for c in conds:
        e = c["prompt_embeds"]
        m = c.get("prompt_embeds_mask")
        if m is None:
            m = torch.ones(e.shape[:2], dtype=torch.long, device=e.device)
        pad = target - e.shape[1]
        if pad > 0:
            e = torch.cat([e, e.new_zeros(e.shape[0], pad, e.shape[2])], dim=1)
            m = torch.cat([m, m.new_zeros(m.shape[0], pad)], dim=1)
        embs.append(e)
        masks.append(m)
    return {"prompt_embeds": torch.cat(embs, dim=0), "prompt_embeds_mask": torch.cat(masks, dim=0)}


def forward_loss(pipe, denoiser, x0, cond, noise, sigma):
    wdtype = next(p for p in denoiser.parameters()).dtype
    device = x0.device
    x0 = x0.to(wdtype)
    noise = noise.to(wdtype)
    bsz, C, _, h, w = x0.shape
    s = sigma.to(device=device, dtype=wdtype).view(-1, 1, 1, 1, 1)
    noisy = (1.0 - s) * x0 + s * noise

    img_shapes = [(1, h // 2, w // 2)] * bsz
    packed = QwenImagePipeline._pack_latents(noisy.permute(0, 2, 1, 3, 4), bsz, C, h, w)

    # cond: batch 1 (broadcast to bsz) or batch == bsz. For per-image captions, build the
    # batched cond with collate_conds() (zero-pad + mask 0, as the official script does).
    emb = cond["prompt_embeds"].to(device=device, dtype=wdtype)
    mask = cond.get("prompt_embeds_mask")
    mask = _mask_or_none(mask.to(device) if mask is not None else None)
    if emb.shape[0] != bsz:
        if emb.shape[0] != 1:
            raise ValueError(
                f"cond batch {emb.shape[0]} != latent batch {bsz}; use collate_conds() to batch per-image conds"
            )
        emb = emb.expand(bsz, -1, -1)
        mask = mask.expand(bsz, -1) if mask is not None else None

    if denoiser.config.guidance_embeds:
        # diffusers' QwenImageTransformer2DModel guidance path is broken (time_text_embed arity);
        # Qwen-Image / Qwen-Image-2512 use guidance_embeds=False.
        raise NotImplementedError("guidance_embeds=True is not supported for qwen_image")
    kwargs = {}

    pred = denoiser(
        hidden_states=packed,
        encoder_hidden_states=emb,
        encoder_hidden_states_mask=mask,
        timestep=sigma.to(device=device, dtype=wdtype),  # == timesteps / 1000
        img_shapes=img_shapes,
        return_dict=False,
        **kwargs,
    )[0]
    vsf = pipe.vae_scale_factor
    pred = QwenImagePipeline._unpack_latents(pred, h * vsf, w * vsf, vsf)
    target = noise - x0
    return torch.mean((pred.float() - target.float()) ** 2)


def save_lora(pipe, lora_state_dict, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    QwenImagePipeline.save_lora_weights(out_dir, transformer_lora_layers=lora_state_dict)
    return os.path.join(out_dir, "pytorch_lora_weights.safetensors")


@torch.no_grad()
def generate(pipe, prompt_cond, negative_cond, height, width, steps, guidance, generator):
    device = pipe._execution_device
    dtype = pipe.transformer.dtype

    def prep(c):
        e = c["prompt_embeds"].to(device=device, dtype=dtype)
        m = c.get("prompt_embeds_mask")
        m = m.to(device) if m is not None else None
        return e, m  # pipeline.encode_prompt drops an all-ones mask itself

    pe, pm = prep(prompt_cond)
    kwargs = dict(prompt_embeds=pe, prompt_embeds_mask=pm)
    if negative_cond is not None and guidance is not None and guidance > 1.0:
        ne, nm = prep(negative_cond)
        kwargs.update(negative_prompt_embeds=ne, negative_prompt_embeds_mask=nm, true_cfg_scale=float(guidance))
    else:
        kwargs["true_cfg_scale"] = 1.0
    if pipe.transformer.config.guidance_embeds:
        raise NotImplementedError("guidance_embeds=True is not supported for qwen_image")
    return pipe(
        height=height, width=width, num_inference_steps=steps, generator=generator,
        max_sequence_length=MAX_SEQUENCE_LENGTH, output_type="pil", **kwargs,
    ).images[0]

In [ ]:
%%writefile lora_adapters/qwen_image21.py
"""Qwen-Image 2.1 adapter.

Mirrors diffusers v0.42.1 examples/dreambooth/train_dreambooth_lora_qwenimage21.py:
  * images are RGBA (this VAE has 4 input channels); RGB inputs get an opaque alpha (= +1 in [-1,1]).
  * latents = (vae.encode(px).latent_dist.sample() - latents_mean) / latents_std, shape (B, z_dim, 1, h, w).
  * x_t = (1 - s) x0 + s noise ; target = noise - x0 ; transformer gets timestep = s (= timesteps / 1000).
  * packing is a plain spatial flatten (patch_size 1); img_shapes = [[(1, h, w)]];
    img_mask = [False * prompt_len, True * (h*w // 4)] ; output tail [-h*w:] is the target prediction.
  * LoRA targets default: to_k, to_q, to_v, to_out.0.
"""
import torch
import torch.nn.functional as F

from diffusers import (
    AutoencoderKLQwenImage21,
    FlowMatchEulerDiscreteScheduler,
    QwenImage21Pipeline,
    QwenImage21Transformer2DModel,
)

FAMILY = "qwen_image21"
PIPELINE_CLASS = QwenImage21Pipeline
DENOISER_ATTR = "transformer"
PREDICTION = "flow"
PIXEL_MULTIPLE = 32  # vae_scale_factor (16) * 2: target latents are grouped into 2x2 vision slots
DEFAULT_STEPS = 40  # pipeline / training-script validation default (overridden by config.sample_sigmas if set)
DEFAULT_GUIDANCE = 1.0  # true_cfg_scale; the model is sampled without CFG (official validation uses 1.0)
LORA_TARGET_MODULES = ["to_k", "to_q", "to_v", "to_out.0"]


def _processor_without_video(model_path: str):
    """Qwen3VLProcessor without its video sub-processor (that one needs torchvision; text prompts never use it)."""
    from transformers import AutoTokenizer, Qwen2VLImageProcessor, Qwen3VLProcessor

    tokenizer = AutoTokenizer.from_pretrained(model_path, subfolder="processor")
    image_processor = Qwen2VLImageProcessor.from_pretrained(model_path, subfolder="processor")
    orig = Qwen3VLProcessor.check_argument_for_proper_class

    def _check(self, name, arg):
        if name == "video_processor" and arg is None:
            return None
        return orig(self, name, arg)

    Qwen3VLProcessor.check_argument_for_proper_class = _check
    try:
        return Qwen3VLProcessor(image_processor=image_processor, tokenizer=tokenizer, video_processor=None,
                                chat_template=tokenizer.chat_template)
    finally:
        Qwen3VLProcessor.check_argument_for_proper_class = orig


def load_pipeline(model_path: str, dtype: torch.dtype):
    try:
        import torchvision  # noqa: F401
        return QwenImage21Pipeline.from_pretrained(model_path, dtype=dtype)
    except ImportError:
        return QwenImage21Pipeline.from_pretrained(model_path, dtype=dtype,
                                                   processor=_processor_without_video(model_path))


def tiny_pipeline(dtype: torch.dtype):
    # Copied from tests/pipelines/qwenimage21/test_qwenimage21.py::get_dummy_components (v0.42.1)
    from transformers import (
        AutoTokenizer,
        Qwen2VLImageProcessor,
        Qwen3VLConfig,
        Qwen3VLForConditionalGeneration,
        Qwen3VLProcessor,
        Qwen3VLVideoProcessor,
    )

    z_dim = 8
    text_hidden_size = 16
    torch.manual_seed(0)
    transformer = QwenImage21Transformer2DModel(
        patch_size=1,
        in_channels=z_dim,
        out_channels=z_dim,
        num_layers=2,
        attention_head_dim=16,
        num_attention_heads=2,
        context_in_dim=text_hidden_size,
        mlp_ratio=2,
        axes_dims_rope=(4, 6, 6),
    )
    torch.manual_seed(0)
    vae = AutoencoderKLQwenImage21(
        base_dim=4,
        decoder_base_dim=4,
        z_dim=z_dim,
        dim_mult=[1, 1, 1, 1, 1],
        num_res_blocks=1,
        attn_scales=[],
        temperal_downsample=[False, True, True, True],
        latents_mean=[0.0] * z_dim,
        latents_std=[1.0] * z_dim,
    )
    torch.manual_seed(0)
    scheduler = FlowMatchEulerDiscreteScheduler()
    torch.manual_seed(0)
    config = Qwen3VLConfig(
        text_config={
            "hidden_size": text_hidden_size,
            "intermediate_size": 16,
            "num_hidden_layers": 2,
            "num_attention_heads": 2,
            "num_key_value_heads": 2,
            "head_dim": 8,
            "rope_parameters": {"rope_type": "default", "rope_theta": 1000000.0, "mrope_section": [1, 1, 2]},
        },
        vision_config={
            "depth": 2,
            "hidden_size": 16,
            "intermediate_size": 16,
            "num_heads": 2,
            "out_hidden_size": text_hidden_size,
            "patch_size": 16,
            "spatial_merge_size": 2,
            "temporal_patch_size": 2,
            "num_position_embeddings": 64,
            "deepstack_visual_indexes": [0],
        },
    )
    text_encoder = Qwen3VLForConditionalGeneration(config).eval()
    tokenizer = AutoTokenizer.from_pretrained("hf-internal-testing/tiny-random-Qwen2VLForConditionalGeneration")
    image_processor = Qwen2VLImageProcessor(
        patch_size=16, merge_size=2, temporal_patch_size=2, min_pixels=32 * 32, max_pixels=64 * 64
    )
    try:
        video_processor = Qwen3VLVideoProcessor(patch_size=16, merge_size=2, temporal_patch_size=2)
    except ImportError:
        # Video processor needs torchvision (absent in the CPU test env); text-only prompts never touch it, so
        # build the processor without it by skipping the class check for that one slot.
        video_processor = None
    if video_processor is not None:
        processor = Qwen3VLProcessor(
            image_processor=image_processor, tokenizer=tokenizer, video_processor=video_processor,
            chat_template=tokenizer.chat_template,
        )
    else:
        orig = Qwen3VLProcessor.check_argument_for_proper_class

        def _check(self, name, arg):
            if name == "video_processor" and arg is None:
                return None
            return orig(self, name, arg)

        Qwen3VLProcessor.check_argument_for_proper_class = _check
        try:
            processor = Qwen3VLProcessor(
                image_processor=image_processor, tokenizer=tokenizer, video_processor=None,
                chat_template=tokenizer.chat_template,
            )
        finally:
            Qwen3VLProcessor.check_argument_for_proper_class = orig
    pipe = QwenImage21Pipeline(
        scheduler=scheduler, vae=vae, text_encoder=text_encoder, processor=processor, transformer=transformer
    )
    return pipe.to(dtype=dtype)


@torch.no_grad()
def encode_prompts(pipe, prompts, device, dtype):
    out = []
    for p in prompts:
        # One prompt at a time -> no padding (left-padding would otherwise change encoder positions).
        emb, mask, _ = pipe.encode_prompt(prompt=[p], device=device)
        if mask is None:
            mask = torch.ones(emb.shape[:2], dtype=torch.long, device=emb.device)
        out.append(
            {
                "prompt_embeds": emb.to(dtype=dtype).cpu(),
                "prompt_embeds_mask": mask.to(torch.long).cpu(),
            }
        )
    return out


@torch.no_grad()
def encode_images(pipe, pixel_values, device, dtype):
    vae = pipe.vae
    px = pixel_values.to(device=device, dtype=vae.dtype)
    if px.shape[1] == 3:  # official script converts every image to RGBA; opaque alpha = 1.0 after normalize
        px = torch.cat([px, torch.ones_like(px[:, :1])], dim=1)
    if px.ndim == 4:
        px = px.unsqueeze(2)  # (B, C, 1, H, W)
    lat = vae.encode(px).latent_dist.sample()
    lat = lat.float()  # normalise with fp32 statistics like the official script
    mean = torch.tensor(vae.config.latents_mean, device=lat.device, dtype=torch.float32).view(1, -1, 1, 1, 1)
    std = torch.tensor(vae.config.latents_std, device=lat.device, dtype=torch.float32).view(1, -1, 1, 1, 1)
    lat = (lat - mean) / std
    return lat.cpu()


def forward_loss(pipe, denoiser, x0, cond, noise, sigma):
    if x0.ndim == 4:
        x0, noise = x0.unsqueeze(2), noise.unsqueeze(2)
    bsz, ch, _, lh, lw = x0.shape
    vsf = pipe.vae_scale_factor
    wdt = next(p for p in denoiser.parameters()).dtype
    dev = x0.device

    s = sigma.to(device=dev, dtype=torch.float32).view(-1, 1, 1, 1, 1)
    x0f, nf = x0.float(), noise.float()
    noisy = ((1.0 - s) * x0f + s * nf).to(wdt)
    target = nf - x0f

    packed = QwenImage21Pipeline._pack_latents(noisy, bsz, ch, lh, lw)
    emb = cond["prompt_embeds"].to(device=dev, dtype=wdt)
    mask = cond.get("prompt_embeds_mask")
    if emb.shape[0] != bsz:
        emb = emb.expand(bsz, -1, -1)
        mask = mask.expand(bsz, -1) if mask is not None else None
    if mask is not None:
        mask = mask.to(dev)
        if mask.all():  # official: no mask when nothing is padded
            mask = None

    img_shapes = [[(1, lh, lw)]] * bsz
    img_mask = torch.cat(
        [
            torch.zeros(bsz, emb.shape[1], dtype=torch.bool, device=dev),
            torch.ones(bsz, (lh * lw) // 4, dtype=torch.bool, device=dev),
        ],
        dim=1,
    )
    pred = denoiser(
        hidden_states=packed,
        encoder_hidden_states=emb,
        encoder_hidden_states_mask=mask,
        timestep=sigma.to(device=dev, dtype=wdt),  # == timesteps / 1000 in the official script
        img_shapes=img_shapes,
        img_mask=img_mask,
        return_dict=False,
    )[0]
    pred = pred[:, -packed.shape[1]:]
    pred = QwenImage21Pipeline._unpack_latents(pred, lh * vsf, lw * vsf, vsf)
    return F.mse_loss(pred.float(), target, reduction="mean")


def save_lora(pipe, lora_state_dict, out_dir):
    import os

    QwenImage21Pipeline.save_lora_weights(out_dir, transformer_lora_layers=lora_state_dict)
    return os.path.join(out_dir, "pytorch_lora_weights.safetensors")


@torch.no_grad()
def generate(pipe, prompt_cond, negative_cond, height, width, steps, guidance, generator):
    dev = pipe._execution_device
    dt = pipe.transformer.dtype

    def prep(c):
        e = c["prompt_embeds"].to(dev, dt)
        m = c.get("prompt_embeds_mask")
        m = m.to(dev) if m is not None else None
        return e, m

    pe, pm = prep(prompt_cond)
    kw = dict(prompt_embeds=pe, prompt_embeds_mask=pm)
    if negative_cond is not None and guidance > 1.0:
        ne, nm = prep(negative_cond)
        kw.update(negative_prompt_embeds=ne, negative_prompt_embeds_mask=nm)
    img = pipe(
        **kw,
        true_cfg_scale=guidance,
        height=height,
        width=width,
        num_inference_steps=steps,
        generator=generator,
        output_type="pil",
    ).images[0]
    if img.mode == "RGBA":
        from PIL import Image
        bg = Image.new("RGB", img.size, (255, 255, 255))
        bg.paste(img, mask=img.getchannel("A"))
        return bg
    return img.convert("RGB")

In [ ]:
%%writefile lora_adapters/flux2_klein.py
"""FLUX.2 [klein] adapter (4B Base by default; works for 9B Base too).

Mirrors diffusers v0.42.1 examples/dreambooth/train_dreambooth_lora_flux2_klein.py:
  - latents = vae.encode(px).latent_dist.mode() -> _patchify_latents (2x2 -> 4C channels)
    -> (x - bn.running_mean) / sqrt(bn.running_var + batch_norm_eps)
  - x_t = (1 - s) * x0 + s * noise, target = noise - x0
  - img_ids = Flux2KleinPipeline._prepare_latent_ids(x0), hidden_states = _pack_latents(x_t)
  - transformer(timestep = timesteps/1000 == sigma, guidance = None unless config.guidance_embeds,
                encoder_hidden_states = prompt_embeds, txt_ids = text_ids, img_ids = img_ids)
  - prediction unpacked with _unpack_latents_with_ids; no attention mask (official script/pipeline pass none;
    prompts are padded to max_sequence_length=512).
"""
import os

import torch
import torch.nn.functional as F
from diffusers import (
    AutoencoderKLFlux2,
    FlowMatchEulerDiscreteScheduler,
    Flux2KleinPipeline,
    Flux2Transformer2DModel,
)

FAMILY = "flux2_klein"
PIPELINE_CLASS = Flux2KleinPipeline
DENOISER_ATTR = "transformer"
PREDICTION = "flow"
PIXEL_MULTIPLE = 16  # vae_scale_factor (8) * 2x2 patchify
DEFAULT_STEPS = 50  # Flux2KleinPipeline.__call__ default (Base model; distilled Klein uses 4 steps / guidance 1.0)
DEFAULT_GUIDANCE = 4.0  # Flux2KleinPipeline.__call__ default (true CFG for the non-distilled Base model)
MAX_SEQUENCE_LENGTH = 512
# Pipeline default hidden-state layers of Qwen3 used for prompt embeds (what the model was trained with and what
# Flux2KleinPipeline uses at inference). NOTE: the official training script's CLI default is [10, 20, 30].
TEXT_ENCODER_OUT_LAYERS = (9, 18, 27)

# Official script default (no --lora_layers): double-block attention + single-block fused qkv/mlp proj + to_out.
LORA_TARGET_MODULES = ["to_k", "to_q", "to_v", "to_out.0", "to_qkv_mlp_proj"] + [
    f"single_transformer_blocks.{i}.attn.to_out" for i in range(24)
]


def _out_layers(pipe):
    return getattr(pipe, "_lvw_text_encoder_out_layers", TEXT_ENCODER_OUT_LAYERS)


def load_pipeline(model_path: str, dtype: torch.dtype):
    return Flux2KleinPipeline.from_pretrained(model_path, torch_dtype=dtype)


def tiny_pipeline(dtype: torch.dtype):
    # Mirrors tests/pipelines/flux2/test_pipeline_flux2_klein.py get_dummy_components (3 stacked layers here,
    # so joint_attention_dim = 3 * hidden_size like the real model).
    from transformers import Qwen2TokenizerFast, Qwen3Config, Qwen3ForCausalLM

    torch.manual_seed(0)
    transformer = Flux2Transformer2DModel(
        patch_size=1,
        in_channels=4,
        num_layers=1,
        num_single_layers=1,
        attention_head_dim=16,
        num_attention_heads=2,
        joint_attention_dim=48,
        timestep_guidance_channels=256,
        axes_dims_rope=[4, 4, 4, 4],
        guidance_embeds=False,
    )
    config = Qwen3Config(
        intermediate_size=16,
        hidden_size=16,
        num_hidden_layers=2,
        num_attention_heads=2,
        num_key_value_heads=2,
        vocab_size=151936,
        max_position_embeddings=1024,
    )
    torch.manual_seed(0)
    text_encoder = Qwen3ForCausalLM(config)
    tokenizer = Qwen2TokenizerFast.from_pretrained("hf-internal-testing/tiny-random-Qwen2VLForConditionalGeneration")
    torch.manual_seed(0)
    vae = AutoencoderKLFlux2(
        sample_size=32,
        in_channels=3,
        out_channels=3,
        down_block_types=("DownEncoderBlock2D",),
        up_block_types=("UpDecoderBlock2D",),
        block_out_channels=(4,),
        layers_per_block=1,
        latent_channels=1,
        norm_num_groups=1,
        use_quant_conv=False,
        use_post_quant_conv=False,
    )
    scheduler = FlowMatchEulerDiscreteScheduler()
    pipe = Flux2KleinPipeline(
        scheduler=scheduler, vae=vae, text_encoder=text_encoder, tokenizer=tokenizer, transformer=transformer
    )
    pipe.to(dtype=dtype)
    pipe._lvw_text_encoder_out_layers = (0, 1, 2)
    return pipe


@torch.no_grad()
def encode_prompts(pipe, prompts, device, dtype):
    out = []
    for p in prompts:
        prompt_embeds, text_ids = pipe.encode_prompt(
            prompt=p,
            device=device,
            max_sequence_length=MAX_SEQUENCE_LENGTH,
            text_encoder_out_layers=_out_layers(pipe),
        )
        out.append({
            "prompt_embeds": prompt_embeds.to(dtype=dtype).cpu(),
            "text_ids": text_ids.cpu(),
        })
    return out


def _bn_stats(vae, device, dtype):
    mean = vae.bn.running_mean.view(1, -1, 1, 1).to(device=device, dtype=dtype)
    std = torch.sqrt(vae.bn.running_var.view(1, -1, 1, 1) + vae.config.batch_norm_eps).to(device=device, dtype=dtype)
    return mean, std


@torch.no_grad()
def encode_images(pipe, pixel_values, device, dtype):
    vae = pipe.vae
    px = pixel_values.to(device=device, dtype=vae.dtype)
    lat = vae.encode(px).latent_dist.mode()
    lat = Flux2KleinPipeline._patchify_latents(lat).float()
    mean, std = _bn_stats(vae, lat.device, torch.float32)
    lat = (lat - mean) / std
    return lat.float().cpu()


def forward_loss(pipe, denoiser, x0, cond, noise, sigma):
    wdtype = next(denoiser.parameters()).dtype
    device = x0.device
    B = x0.shape[0]
    s = sigma.to(device=device, dtype=torch.float32).view(-1, 1, 1, 1)
    x0f, noisef = x0.float(), noise.float()
    noisy = (1.0 - s) * x0f + s * noisef

    img_ids = Flux2KleinPipeline._prepare_latent_ids(x0).to(device=device)
    packed = Flux2KleinPipeline._pack_latents(noisy).to(wdtype)

    prompt_embeds = cond["prompt_embeds"].to(device=device, dtype=wdtype)
    text_ids = cond["text_ids"].to(device=device)
    if prompt_embeds.shape[0] != B:
        prompt_embeds = prompt_embeds.expand(B, -1, -1)
        text_ids = text_ids.expand(B, -1, -1)

    if denoiser.config.guidance_embeds:
        guidance = torch.full((B,), 3.5, device=device, dtype=wdtype)  # official --guidance_scale default
    else:
        guidance = None

    # official: timestep = timesteps / 1000, with timesteps = sigma * num_train_timesteps (1000)
    timestep = sigma.to(device=device, dtype=wdtype).view(-1).expand(B)
    model_pred = denoiser(
        hidden_states=packed,
        timestep=timestep,
        guidance=guidance,
        encoder_hidden_states=prompt_embeds,
        txt_ids=text_ids,
        img_ids=img_ids,
        return_dict=False,
    )[0]
    model_pred = model_pred[:, : packed.size(1)]
    model_pred = Flux2KleinPipeline._unpack_latents_with_ids(model_pred, img_ids)

    target = noisef - x0f
    return F.mse_loss(model_pred.float(), target, reduction="mean")


def save_lora(pipe, lora_state_dict, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    Flux2KleinPipeline.save_lora_weights(
        save_directory=out_dir, transformer_lora_layers=lora_state_dict, safe_serialization=True
    )
    return os.path.join(out_dir, "pytorch_lora_weights.safetensors")


@torch.no_grad()
def generate(pipe, prompt_cond, negative_cond, height, width, steps, guidance, generator):
    dev = pipe._execution_device
    dt = pipe.transformer.dtype
    kwargs = dict(
        prompt_embeds=prompt_cond["prompt_embeds"].to(device=dev, dtype=dt),
        height=height,
        width=width,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=generator,
        text_encoder_out_layers=_out_layers(pipe),
        output_type="pil",
    )
    # With guidance > 1 on the Base model the pipeline runs true CFG; passing cached negative embeds avoids
    # running the text encoder for the empty negative prompt.
    if negative_cond is not None:
        kwargs["negative_prompt_embeds"] = negative_cond["prompt_embeds"].to(device=dev, dtype=dt)
    elif guidance is not None and guidance > 1 and not getattr(pipe.config, "is_distilled", False):
        # pipeline_flux2_klein.py: do_classifier_free_guidance -> encode_prompt("") runs the Qwen3 text encoder.
        te = getattr(pipe, "text_encoder", None)
        te_dev = next(te.parameters()).device if te is not None else None
        if te is None or te_dev != torch.device(dev):
            raise ValueError(
                "flux2_klein.generate: Base (non-distilled) model with guidance > 1 runs true CFG and needs "
                "negative embeddings. Pass negative_cond=encode_prompts(pipe, ['']) (cached while the text "
                f"encoder was on the device); text encoder is {'missing' if te is None else 'on ' + str(te_dev)}."
            )
    return pipe(**kwargs).images[0]

In [ ]:
%%writefile lora_adapters/z_image.py
"""Z-Image (Tongyi-MAI/Z-Image, Z-Image-Turbo) adapter.

Mirrors diffusers v0.42.1 examples/dreambooth/train_dreambooth_lora_z_image.py:
  latents   = (vae.encode(x).latent_dist.mode() - shift_factor) * scaling_factor
  x_t       = (1 - sigma) * x0 + sigma * noise
  timestep  = (1000 - sigma*1000) / 1000 = 1 - sigma   (Z-Image's "t" runs 0=noise -> 1=clean)
  denoiser  = transformer(list[(C,1,H,W)], t, list[(L_i, D)])  -> list[(C,1,H,W)]
  model_pred = -stack(out).squeeze(2)          (Z-Image negates the prediction)
  target    = noise - x0
LoRA targets: ["to_k", "to_q", "to_v", "to_out.0"] (script default).
Text conditioning: Qwen3 hidden_states[-2] after chat template, with padding tokens REMOVED
(variable-length list, no attention mask needed by the transformer).
"""
import os

import torch
import torch.nn.functional as F
from diffusers import (
    AutoencoderKL,
    FlowMatchEulerDiscreteScheduler,
    ZImagePipeline,
    ZImageTransformer2DModel,
)

FAMILY = "z_image"
PIPELINE_CLASS = ZImagePipeline
DENOISER_ATTR = "transformer"
PREDICTION = "flow"
PIXEL_MULTIPLE = 16  # vae_scale_factor (8) * 2 (patch size)
# ZImagePipeline.__call__ defaults (official validation calls the pipeline with defaults; base Z-Image).
# For Z-Image-Turbo use steps=8..9, guidance=0.0.
DEFAULT_STEPS = 50
DEFAULT_GUIDANCE = 5.0
LORA_TARGET_MODULES = ["to_k", "to_q", "to_v", "to_out.0"]
MAX_SEQUENCE_LENGTH = 512  # train script default --max_sequence_length


def load_pipeline(model_path: str, dtype: torch.dtype):
    return ZImagePipeline.from_pretrained(model_path, torch_dtype=dtype)


def tiny_pipeline(dtype: torch.dtype):
    # Copied from diffusers v0.42.1 tests/pipelines/z_image/test_z_image.py get_dummy_components()
    from transformers import Qwen2Tokenizer, Qwen3Config, Qwen3Model

    torch.manual_seed(0)
    transformer = ZImageTransformer2DModel(
        all_patch_size=(2,),
        all_f_patch_size=(1,),
        in_channels=16,
        dim=32,
        n_layers=2,
        n_refiner_layers=1,
        n_heads=2,
        n_kv_heads=2,
        norm_eps=1e-5,
        qk_norm=True,
        cap_feat_dim=16,
        rope_theta=256.0,
        t_scale=1000.0,
        axes_dims=[8, 4, 4],
        axes_lens=[256, 32, 32],
    )
    with torch.no_grad():
        transformer.x_pad_token.copy_(torch.ones_like(transformer.x_pad_token.data))
        transformer.cap_pad_token.copy_(torch.ones_like(transformer.cap_pad_token.data))

    torch.manual_seed(0)
    vae = AutoencoderKL(
        in_channels=3,
        out_channels=3,
        down_block_types=["DownEncoderBlock2D", "DownEncoderBlock2D"],
        up_block_types=["UpDecoderBlock2D", "UpDecoderBlock2D"],
        block_out_channels=[32, 64],
        layers_per_block=1,
        latent_channels=16,
        norm_num_groups=32,
        sample_size=32,
        scaling_factor=0.3611,
        shift_factor=0.1159,
    )
    torch.manual_seed(0)
    scheduler = FlowMatchEulerDiscreteScheduler()
    torch.manual_seed(0)
    config = Qwen3Config(
        hidden_size=16,
        intermediate_size=16,
        num_hidden_layers=2,
        num_attention_heads=2,
        num_key_value_heads=2,
        vocab_size=151936,
        max_position_embeddings=512,
    )
    text_encoder = Qwen3Model(config)
    tokenizer = Qwen2Tokenizer.from_pretrained("hf-internal-testing/tiny-random-Qwen2VLForConditionalGeneration")
    pipe = ZImagePipeline(
        scheduler=scheduler, vae=vae, text_encoder=text_encoder, tokenizer=tokenizer, transformer=transformer
    )
    return pipe.to(dtype=dtype)


@torch.no_grad()
def encode_prompts(pipe, prompts, device, dtype):
    out = []
    for p in prompts:
        # same call as compute_text_embeddings() in the official script (one prompt at a time -> no padding)
        embeds = pipe._encode_prompt(prompt=[p], device=device, max_sequence_length=MAX_SEQUENCE_LENGTH)
        e = embeds[0].to(dtype=dtype).detach().cpu()  # (L, D), padding already stripped
        out.append({"prompt_embeds": e.unsqueeze(0)})  # (1, L, D)
    return out


@torch.no_grad()
def encode_images(pipe, pixel_values, device, dtype):
    vae = pipe.vae
    px = pixel_values.to(device=device, dtype=vae.dtype)
    lat = vae.encode(px).latent_dist.mode()
    lat = (lat.float() - vae.config.shift_factor) * vae.config.scaling_factor
    return lat.float().cpu()


def _embeds_list(cond, device, dtype, bsz):
    pe = cond["prompt_embeds"].to(device=device, dtype=dtype)  # (1, L, D)
    lst = [pe[0]]
    if bsz > 1:
        lst = lst * bsz
    return lst


def forward_loss(pipe, denoiser, x0, cond, noise, sigma):
    device = x0.device
    wdtype = next(p for p in denoiser.parameters()).dtype
    x0 = x0.float()
    noise = noise.float()
    s = sigma.to(device=device, dtype=torch.float32).view(-1, *([1] * (x0.ndim - 1)))
    noisy = (1.0 - s) * x0 + s * noise
    # official: timestep_normalized = (1000 - timesteps) / 1000 with timesteps = sigma * 1000
    t = (1000.0 - sigma.to(device=device, dtype=torch.float32) * 1000.0) / 1000.0
    bsz = x0.shape[0]
    t = t.expand(bsz) if t.numel() == 1 else t
    inp = list(noisy.to(wdtype).unsqueeze(2).unbind(dim=0))  # list of (C,1,H,W)
    # timestep stays fp32 like the official script / pipeline (bf16 would bin t*1000 coarsely)
    out = denoiser(inp, t.float(), _embeds_list(cond, device, wdtype, bsz), return_dict=False)[0]
    pred = -torch.stack(out, dim=0).squeeze(2).float()  # Z-Image negates the prediction
    target = noise - x0
    return F.mse_loss(pred, target, reduction="mean")


def save_lora(pipe, lora_state_dict, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    kwargs = {}
    den = getattr(pipe, DENOISER_ATTR)
    cfg = getattr(den, "peft_config", None)
    if cfg and "default" in cfg:
        kwargs["transformer_lora_adapter_metadata"] = cfg["default"].to_dict()
    ZImagePipeline.save_lora_weights(
        out_dir, transformer_lora_layers=lora_state_dict, safe_serialization=True, **kwargs
    )
    return os.path.join(out_dir, "pytorch_lora_weights.safetensors")


@torch.no_grad()
def generate(pipe, prompt_cond, negative_cond, height, width, steps, guidance, generator):
    device = pipe._execution_device
    dtype = pipe.transformer.dtype
    pe = [prompt_cond["prompt_embeds"].to(device=device, dtype=dtype)[0]]
    kwargs = {}
    if guidance and guidance > 0:
        if negative_cond is not None:
            kwargs["negative_prompt_embeds"] = [negative_cond["prompt_embeds"].to(device=device, dtype=dtype)[0]]
        elif getattr(pipe, "text_encoder", None) is not None:
            kwargs["negative_prompt_embeds"] = [
                e.to(device=device, dtype=dtype)
                for e in pipe._encode_prompt(prompt=[""], device=device, max_sequence_length=MAX_SEQUENCE_LENGTH)
            ]
        else:
            guidance = 0.0  # no negative embeds and no text encoder -> CFG impossible
    return pipe(
        prompt_embeds=pe,
        height=height,
        width=width,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=generator,
        output_type="pil",
        **kwargs,
    ).images[0]

In [ ]:
%%writefile lora_adapters/krea2.py
"""Krea 2 adapter (diffusers 0.42.1).

Mirrors examples/dreambooth/train_dreambooth_lora_krea2.py at tag v0.42.1:
  * latents: vae.encode(pixels[:, :, None]).latent_dist.sample(), then (z - latents_mean) / latents_std
    (the script multiplies by 1/latents_std), kept 5D (B, C, 1, h, w)
  * flow matching: x_t = (1 - s) * x0 + s * noise, target = noise - x0
  * timestep passed to the transformer = timesteps / 1000 = sigma in [0, 1] (the transformer scales by 1e3 itself)
  * 2x2 patch packing -> (B, (h/2)*(w/2), C*4); position_ids from Krea2Pipeline.prepare_position_ids
  * text: stacked Qwen3-VL hidden states (B, 512, num_text_layers, D) + bool mask, fixed length 512
  * LoRA target modules: the script's default list (Krea 2 authors' recommended config)
"""
import torch

from diffusers import (
    AutoencoderKLQwenImage,
    FlowMatchEulerDiscreteScheduler,
    Krea2Pipeline,
    Krea2Transformer2DModel,
)

FAMILY = "krea2"
PIPELINE_CLASS = Krea2Pipeline
DENOISER_ATTR = "transformer"
PREDICTION = "flow"
PIXEL_MULTIPLE = 16  # vae_scale_factor (8) * patch_size (2) for the real model
DEFAULT_STEPS = 28  # pipeline default for the base checkpoint (turbo/distilled: 8 steps, guidance 0.0)
DEFAULT_GUIDANCE = 4.5  # Krea 2 convention: cond + g * (cond - uncond); 0 disables CFG
MAX_SEQUENCE_LENGTH = 512  # official --max_sequence_length default

LORA_TARGET_MODULES = [
    "img_in",
    "final_layer.linear",
    "to_q",
    "to_k",
    "to_v",
    "to_out.0",
    "to_gate",
    "ff.up",
    "ff.down",
    "text_fusion.projector",
    "txt_in.linear_1",
    "txt_in.linear_2",
    "time_embed.linear_1",
    "time_embed.linear_2",
    "time_mod_proj",
]


def load_pipeline(model_path: str, dtype: torch.dtype):
    # Works for a local diffusers-format dir (model_index.json) or a HF repo id.
    return Krea2Pipeline.from_pretrained(model_path, torch_dtype=dtype)


def tiny_pipeline(dtype: torch.dtype):
    # Copied from tests/pipelines/krea2/test_krea2.py::get_dummy_components (v0.42.1).
    from transformers import Qwen2Tokenizer, Qwen3VLConfig, Qwen3VLModel

    torch.manual_seed(0)
    transformer = Krea2Transformer2DModel(
        in_channels=16,
        num_layers=2,
        attention_head_dim=8,
        num_attention_heads=4,
        num_key_value_heads=2,
        intermediate_size=32,
        timestep_embed_dim=8,
        text_hidden_dim=16,
        num_text_layers=3,
        text_num_attention_heads=2,
        text_num_key_value_heads=1,
        text_intermediate_size=16,
        num_layerwise_text_blocks=1,
        num_refiner_text_blocks=1,
        axes_dims_rope=(4, 2, 2),
        rope_theta=1000.0,
    )
    torch.manual_seed(0)
    z_dim = 4
    vae = AutoencoderKLQwenImage(
        base_dim=z_dim * 6,
        z_dim=z_dim,
        dim_mult=[1, 2, 4],
        num_res_blocks=1,
        temperal_downsample=[False, True],
        latents_mean=[0.0] * 4,
        latents_std=[1.0] * 4,
    )
    scheduler = FlowMatchEulerDiscreteScheduler(
        use_dynamic_shifting=True,
        base_shift=0.5,
        max_shift=1.15,
        base_image_seq_len=256,
        max_image_seq_len=6400,
    )
    torch.manual_seed(0)
    config = Qwen3VLConfig(
        text_config={
            "hidden_size": 16,
            "intermediate_size": 16,
            "num_hidden_layers": 2,
            "num_attention_heads": 2,
            "num_key_value_heads": 2,
            "head_dim": 8,
        },
        vision_config={
            "depth": 2,
            "hidden_size": 16,
            "intermediate_size": 16,
            "num_heads": 2,
            "out_hidden_size": 16,
        },
        vocab_size=152064,
    )
    text_encoder = Qwen3VLModel(config).eval()
    tokenizer = Qwen2Tokenizer.from_pretrained("hf-internal-testing/tiny-random-Qwen2VLForConditionalGeneration")
    pipe = Krea2Pipeline(
        scheduler=scheduler,
        vae=vae,
        text_encoder=text_encoder,
        tokenizer=tokenizer,
        transformer=transformer,
        text_encoder_select_layers=(0, 1, 2),
    )
    return pipe if dtype == torch.float32 else pipe.to(dtype=dtype)


@torch.no_grad()
def encode_prompts(pipe, prompts, device, dtype):
    te = pipe.text_encoder
    orig = next(te.parameters()).device
    te.to(device)
    out = []
    for p in prompts:
        emb, mask = pipe.encode_prompt(prompt=[p], device=device, max_sequence_length=MAX_SEQUENCE_LENGTH)
        out.append({
            "prompt_embeds": emb.to(dtype).cpu(),
            "prompt_embeds_mask": mask.bool().cpu(),
        })
    te.to(orig)
    return out


def _latent_stats(vae, device, dtype):
    z = vae.config.z_dim
    mean = torch.tensor(vae.config.latents_mean).view(1, z, 1, 1, 1).to(device, dtype)
    inv_std = 1.0 / torch.tensor(vae.config.latents_std).view(1, z, 1, 1, 1).to(device, dtype)
    return mean, inv_std


@torch.no_grad()
def encode_images(pipe, pixel_values, device, dtype):
    vae = pipe.vae
    px = pixel_values.to(device=device, dtype=vae.dtype)
    if px.ndim == 4:
        px = px.unsqueeze(2)  # (B, 3, 1, H, W) as in the official script's collate_fn
    lat = vae.encode(px).latent_dist.sample()
    mean, inv_std = _latent_stats(vae, lat.device, torch.float32)
    lat = (lat.float() - mean) * inv_std
    return lat.float().cpu()  # (B, C, 1, H/8, W/8)


def _pack(x, p=2):
    # (B, C, 1, H, W) -> (B, (H/p)*(W/p), C*p*p), identical to the script's inlined Krea2Pipeline._pack_latents
    b, c, _, h, w = x.shape
    x = x.permute(0, 2, 1, 3, 4).reshape(b, c, h // p, p, w // p, p)
    x = x.permute(0, 2, 4, 1, 3, 5)
    return x.reshape(b, (h // p) * (w // p), c * p * p)


def _unpack(x, h, w, p=2):
    b, _, ch = x.shape
    x = x.view(b, h // p, w // p, ch // (p * p), p, p)
    x = x.permute(0, 3, 1, 4, 2, 5)
    return x.reshape(b, ch // (p * p), 1, h, w)


def forward_loss(pipe, denoiser, x0, cond, noise, sigma):
    p = getattr(pipe, "patch_size", 2)
    wdtype = denoiser.img_in.weight.dtype  # norms stay fp32 via _keep_in_fp32_modules
    if x0.ndim == 4:
        x0, noise = x0.unsqueeze(2), noise.unsqueeze(2)
    b = x0.shape[0]
    s = sigma.to(x0.device, torch.float32).view(b, 1, 1, 1, 1)
    x_t = ((1.0 - s) * x0.float() + s * noise.float()).to(wdtype)
    h, w = x0.shape[-2], x0.shape[-1]

    emb = cond["prompt_embeds"].to(x0.device, wdtype)
    mask = cond["prompt_embeds_mask"].to(x0.device).bool()
    if emb.shape[0] != b:
        emb = emb.expand(b, *emb.shape[1:])
        mask = mask.expand(b, *mask.shape[1:])
    position_ids = Krea2Pipeline.prepare_position_ids(emb.shape[1], h // p, w // p, x0.device)

    pred = denoiser(
        hidden_states=_pack(x_t, p),
        encoder_hidden_states=emb,
        timestep=sigma.to(x0.device, torch.float32).view(b),  # official: fp32 timesteps / 1000 == sigma
        position_ids=position_ids,
        encoder_attention_mask=mask,
        return_dict=False,
    )[0]
    pred = _unpack(pred, h, w, p)
    target = noise.float() - x0.float()
    return torch.mean((pred.float() - target) ** 2)


def save_lora(pipe, lora_state_dict, out_dir):
    import os

    Krea2Pipeline.save_lora_weights(out_dir, transformer_lora_layers=lora_state_dict)
    return os.path.join(out_dir, "pytorch_lora_weights.safetensors")


@torch.no_grad()
def generate(pipe, prompt_cond, negative_cond, height, width, steps, guidance, generator):
    dev = pipe._execution_device
    dt = pipe.transformer.dtype
    kw = dict(
        prompt_embeds=prompt_cond["prompt_embeds"].to(dev, dt),
        prompt_embeds_mask=prompt_cond["prompt_embeds_mask"].to(dev),
    )
    if guidance > 0:
        if negative_cond is not None:
            kw["negative_prompt_embeds"] = negative_cond["prompt_embeds"].to(dev, dt)
            kw["negative_prompt_embeds_mask"] = negative_cond["prompt_embeds_mask"].to(dev)
        elif getattr(pipe, "text_encoder", None) is None:
            raise ValueError("Krea 2 CFG needs negative embeddings (encode '' with encode_prompts) when no text encoder.")
        # else: pipeline encodes "" as the negative prompt itself
    return pipe(
        height=height,
        width=width,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=generator,
        max_sequence_length=kw["prompt_embeds"].shape[1],
        output_type="pil",
        **kw,
    ).images[0]

In [ ]:
%%writefile lora_adapters/flux1.py
"""FLUX.1 (dev / schnell) adapter, mirroring diffusers v0.42.1
examples/dreambooth/train_dreambooth_lora_flux.py."""
import os

import torch
from diffusers import (
    AutoencoderKL,
    FlowMatchEulerDiscreteScheduler,
    FluxPipeline,
    FluxTransformer2DModel,
)

FAMILY = "flux1"
PIPELINE_CLASS = FluxPipeline
DENOISER_ATTR = "transformer"
PREDICTION = "flow"
PIXEL_MULTIPLE = 16          # vae_scale_factor (8) * 2 (2x2 latent packing)
DEFAULT_STEPS = 28           # FluxPipeline.__call__ default (FLUX.1-dev). schnell: 4
DEFAULT_GUIDANCE = 3.5       # distilled guidance for FLUX.1-dev. schnell: 0.0 (guidance_embeds=False -> ignored)
TRAIN_GUIDANCE = 3.5         # official --guidance_scale default, used only when transformer.config.guidance_embeds
MAX_SEQUENCE_LENGTH = 512    # official --max_sequence_length default (schnell supports at most 256)
LORA_TARGET_MODULES = [
    "attn.to_k",
    "attn.to_q",
    "attn.to_v",
    "attn.to_out.0",
    "attn.add_k_proj",
    "attn.add_q_proj",
    "attn.add_v_proj",
    "attn.to_add_out",
    "ff.net.0.proj",
    "ff.net.2",
    "ff_context.net.0.proj",
    "ff_context.net.2",
]


def _max_seq_len(pipe):
    # schnell has no guidance embedding and was trained with 256 T5 tokens
    if not pipe.transformer.config.guidance_embeds:
        return 256
    return MAX_SEQUENCE_LENGTH


def load_pipeline(model_path: str, dtype: torch.dtype):
    # works for a local diffusers-format dir (model_index.json) or a HF repo id
    return FluxPipeline.from_pretrained(model_path, torch_dtype=dtype)


def tiny_pipeline(dtype: torch.dtype):
    """Same component classes as the real checkpoint, shrunk (copied from tests/pipelines/flux get_dummy_components).
    guidance_embeds=True so the FLUX.1-dev guidance path is exercised."""
    from transformers import AutoConfig, AutoTokenizer, CLIPTextConfig, CLIPTextModel, CLIPTokenizer, T5EncoderModel

    torch.manual_seed(0)
    transformer = FluxTransformer2DModel(
        patch_size=1,
        in_channels=4,
        num_layers=1,
        num_single_layers=1,
        attention_head_dim=16,
        num_attention_heads=2,
        joint_attention_dim=32,
        pooled_projection_dim=32,
        axes_dims_rope=[4, 4, 8],
        guidance_embeds=True,
    )
    clip_cfg = CLIPTextConfig(
        bos_token_id=0, eos_token_id=2, hidden_size=32, intermediate_size=37, layer_norm_eps=1e-05,
        num_attention_heads=4, num_hidden_layers=5, pad_token_id=1, vocab_size=1000, hidden_act="gelu",
        projection_dim=32,
    )
    torch.manual_seed(0)
    text_encoder = CLIPTextModel(clip_cfg)
    torch.manual_seed(0)
    t5_cfg = AutoConfig.from_pretrained("hf-internal-testing/tiny-random-t5")
    text_encoder_2 = T5EncoderModel(t5_cfg)
    tokenizer = CLIPTokenizer.from_pretrained("hf-internal-testing/tiny-random-clip")
    tokenizer_2 = AutoTokenizer.from_pretrained("hf-internal-testing/tiny-random-t5")
    torch.manual_seed(0)
    vae = AutoencoderKL(
        sample_size=32, in_channels=3, out_channels=3, block_out_channels=(4,), layers_per_block=1,
        latent_channels=1, norm_num_groups=1, use_quant_conv=False, use_post_quant_conv=False,
        shift_factor=0.0609, scaling_factor=1.5035,
    )
    scheduler = FlowMatchEulerDiscreteScheduler()
    pipe = FluxPipeline(
        scheduler=scheduler, vae=vae, text_encoder=text_encoder, tokenizer=tokenizer,
        text_encoder_2=text_encoder_2, tokenizer_2=tokenizer_2, transformer=transformer,
        image_encoder=None, feature_extractor=None,
    )
    return pipe.to(dtype=dtype)


@torch.no_grad()
def encode_prompts(pipe, prompts, device, dtype):
    """One dict per prompt: prompt_embeds (1,L,D) T5, pooled_prompt_embeds (1,D') CLIP pooled, text_ids (1,L,3)."""
    for m in (pipe.text_encoder, pipe.text_encoder_2):
        m.to(device)
    out = []
    for p in prompts:
        pe, pooled, text_ids = pipe.encode_prompt(
            prompt=[p], prompt_2=None, device=device, num_images_per_prompt=1,
            max_sequence_length=_max_seq_len(pipe),
        )
        out.append({
            "prompt_embeds": pe.to(dtype).cpu(),
            "pooled_prompt_embeds": pooled.to(dtype).cpu(),
            "text_ids": text_ids.to(dtype).unsqueeze(0).cpu(),  # stored with batch dim 1; squeezed for the model
        })
    return out


@torch.no_grad()
def encode_images(pipe, pixel_values, device, dtype):
    vae = pipe.vae
    vae.to(device)
    lat = vae.encode(pixel_values.to(device=device, dtype=vae.dtype)).latent_dist.sample()
    lat = (lat - vae.config.shift_factor) * vae.config.scaling_factor
    return lat.float().cpu()


def _txt_ids(cond, device, dtype):
    t = cond["text_ids"]
    if t.ndim == 3:
        t = t[0]
    return t.to(device=device, dtype=dtype)


def forward_loss(pipe, denoiser, x0, cond, noise, sigma):
    dtype = next(denoiser.parameters()).dtype
    device = x0.device
    B, C, h, w = x0.shape
    vae_scale_factor = pipe.vae_scale_factor

    s = sigma.to(device=device, dtype=torch.float32).view(-1, 1, 1, 1)
    noisy = (1.0 - s) * x0.float() + s * noise.float()
    noisy = noisy.to(dtype)

    packed = FluxPipeline._pack_latents(noisy, batch_size=B, num_channels_latents=C, height=h, width=w)
    img_ids = FluxPipeline._prepare_latent_image_ids(B, h // 2, w // 2, device, dtype)

    if denoiser.config.guidance_embeds:
        guidance = torch.tensor([TRAIN_GUIDANCE], device=device, dtype=torch.float32).expand(B)
    else:
        guidance = None

    pe = cond["prompt_embeds"].to(device=device, dtype=dtype)
    pooled = cond["pooled_prompt_embeds"].to(device=device, dtype=dtype)
    if pe.shape[0] != B:
        pe = pe.expand(B, -1, -1)
        pooled = pooled.expand(B, -1)

    # official: timestep = timesteps / 1000 == sigma (transformer multiplies by 1000 internally)
    timestep = sigma.to(device=device, dtype=dtype).view(-1).expand(B)
    model_pred = denoiser(
        hidden_states=packed,
        timestep=timestep,
        guidance=guidance,
        pooled_projections=pooled,
        encoder_hidden_states=pe,
        txt_ids=_txt_ids(cond, device, dtype),
        img_ids=img_ids,
        return_dict=False,
    )[0]
    model_pred = FluxPipeline._unpack_latents(
        model_pred, height=h * vae_scale_factor, width=w * vae_scale_factor, vae_scale_factor=vae_scale_factor
    )
    target = noise.float() - x0.float()
    return torch.mean((model_pred.float() - target) ** 2)


def save_lora(pipe, lora_state_dict, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    FluxPipeline.save_lora_weights(out_dir, transformer_lora_layers=lora_state_dict, safe_serialization=True)
    return os.path.join(out_dir, "pytorch_lora_weights.safetensors")


@torch.no_grad()
def generate(pipe, prompt_cond, negative_cond, height, width, steps, guidance, generator):
    """guidance -> FLUX.1 distilled guidance_scale (ignored by schnell). negative_cond is passed through but only
    used if true_cfg_scale > 1, which the official validation does not do (true_cfg_scale=1.0)."""
    dev = pipe._execution_device
    dtype = pipe.transformer.dtype
    kw = dict(
        prompt_embeds=prompt_cond["prompt_embeds"].to(dev, dtype),
        pooled_prompt_embeds=prompt_cond["pooled_prompt_embeds"].to(dev, dtype),
    )
    if negative_cond is not None:
        kw["negative_prompt_embeds"] = negative_cond["prompt_embeds"].to(dev, dtype)
        kw["negative_pooled_prompt_embeds"] = negative_cond["pooled_prompt_embeds"].to(dev, dtype)
    out = pipe(
        height=height, width=width, num_inference_steps=steps, guidance_scale=guidance,
        true_cfg_scale=1.0, generator=generator, max_sequence_length=_max_seq_len(pipe), **kw,
    )
    return out.images[0]

In [ ]:
%%writefile lora_adapters/chroma.py
"""Chroma (lodestones/Chroma1-HD) adapter.

diffusers 0.42.1 ships NO official Chroma training script; this follows
examples/dreambooth/train_dreambooth_lora_flux.py (Chroma is a pruned Flux-schnell derivative)
combined with ChromaPipeline.__call__ for the Chroma-specific bits:
  * T5 only (no CLIP pooled embeds), T5 run WITH the tokenizer attention mask,
  * text mask keeps the real tokens + the first padding token, extended with ones for image tokens,
  * no guidance embedding (Chroma's distilled_guidance_layer is fed timestep only), real CFG at inference.
"""
import os

import torch
import torch.nn.functional as F
from diffusers import (
    AutoencoderKL,
    ChromaPipeline,
    ChromaTransformer2DModel,
    FlowMatchEulerDiscreteScheduler,
)

FAMILY = "chroma"
PIPELINE_CLASS = ChromaPipeline
DENOISER_ATTR = "transformer"
PREDICTION = "flow"
PIXEL_MULTIPLE = 16          # vae 8x * 2x2 packing
DEFAULT_STEPS = 35           # ChromaPipeline.__call__ default
DEFAULT_GUIDANCE = 5.0       # ChromaPipeline.__call__ default (true CFG with negative prompt)
MAX_SEQ_LEN = 512
# Default target modules of train_dreambooth_lora_flux.py (same module names exist in ChromaTransformer2DModel)
LORA_TARGET_MODULES = [
    "attn.to_k",
    "attn.to_q",
    "attn.to_v",
    "attn.to_out.0",
    "attn.add_k_proj",
    "attn.add_q_proj",
    "attn.add_v_proj",
    "attn.to_add_out",
    "ff.net.0.proj",
    "ff.net.2",
    "ff_context.net.0.proj",
    "ff_context.net.2",
]


def load_pipeline(model_path: str, dtype: torch.dtype):
    # works for a local diffusers-format dir (model_index.json) or a hub repo id
    return ChromaPipeline.from_pretrained(model_path, torch_dtype=dtype)


def tiny_pipeline(dtype: torch.dtype):
    from transformers import AutoConfig, AutoTokenizer, T5EncoderModel

    torch.manual_seed(0)
    transformer = ChromaTransformer2DModel(
        patch_size=1,
        in_channels=4,
        num_layers=1,
        num_single_layers=1,
        attention_head_dim=16,
        num_attention_heads=2,
        joint_attention_dim=32,
        axes_dims_rope=[4, 4, 8],
        approximator_hidden_dim=32,
        approximator_layers=1,
        approximator_num_channels=16,
    )
    torch.manual_seed(0)
    config = AutoConfig.from_pretrained("hf-internal-testing/tiny-random-t5")
    text_encoder = T5EncoderModel(config).eval()
    tokenizer = AutoTokenizer.from_pretrained("hf-internal-testing/tiny-random-t5")
    torch.manual_seed(0)
    vae = AutoencoderKL(
        sample_size=32,
        in_channels=3,
        out_channels=3,
        block_out_channels=(4,),
        layers_per_block=1,
        latent_channels=1,
        norm_num_groups=1,
        use_quant_conv=False,
        use_post_quant_conv=False,
        shift_factor=0.0609,
        scaling_factor=1.5035,
    )
    pipe = ChromaPipeline(
        scheduler=FlowMatchEulerDiscreteScheduler(),
        vae=vae,
        text_encoder=text_encoder,
        tokenizer=tokenizer,
        transformer=transformer,
        image_encoder=None,
        feature_extractor=None,
    )
    return pipe.to(dtype=dtype)


@torch.no_grad()
def encode_prompts(pipe, prompts, device, dtype):
    out = []
    device = torch.device(device)
    if next(pipe.text_encoder.parameters()).device != device:
        pipe.text_encoder.to(device)
    for p in prompts:
        # pipeline helper: padding="max_length", T5 run with tokenizer mask, chroma text mask (real + 1st pad)
        embeds, mask = pipe._get_t5_prompt_embeds(
            prompt=[p], num_images_per_prompt=1, max_sequence_length=MAX_SEQ_LEN, device=device, dtype=dtype
        )
        out.append({
            "prompt_embeds": embeds.detach().to("cpu"),
            "prompt_attention_mask": mask.detach().to("cpu"),
        })
    return out


@torch.no_grad()
def encode_images(pipe, pixel_values, device, dtype):
    vae = pipe.vae
    px = pixel_values.to(device=device, dtype=vae.dtype)
    lat = vae.encode(px).latent_dist.sample()
    lat = (lat - vae.config.shift_factor) * vae.config.scaling_factor  # as in train_dreambooth_lora_flux.py
    return lat.to("cpu", torch.float32)


def forward_loss(pipe, denoiser, x0, cond, noise, sigma):
    wdtype = denoiser.dtype
    B, C, h, w = x0.shape
    device = x0.device
    vsf = pipe.vae_scale_factor
    s = sigma.to(device=device, dtype=x0.dtype).view(-1, 1, 1, 1)
    noisy = (1.0 - s) * x0 + s * noise

    packed = ChromaPipeline._pack_latents(noisy, B, C, h, w).to(wdtype)
    img_ids = ChromaPipeline._prepare_latent_image_ids(B, h // 2, w // 2, device, wdtype)

    prompt_embeds = cond["prompt_embeds"].to(device=device, dtype=wdtype)
    pmask = cond["prompt_attention_mask"].to(device=device)
    if prompt_embeds.shape[0] != B:
        prompt_embeds = prompt_embeds.expand(B, -1, -1)
        pmask = pmask.expand(B, -1)
    txt_ids = torch.zeros(prompt_embeds.shape[1], 3, device=device, dtype=wdtype)
    # extend the text mask with ones for the image tokens (ChromaPipeline._prepare_attention_mask)
    attn_mask = torch.cat([pmask.to(wdtype), torch.ones(B, packed.shape[1], device=device, dtype=wdtype)], dim=1)

    timesteps = sigma.to(device) * 1000.0  # scheduler timesteps; denoiser receives t/1000 (= sigma)
    pred = denoiser(
        hidden_states=packed,
        timestep=(timesteps / 1000).to(wdtype),
        encoder_hidden_states=prompt_embeds,
        txt_ids=txt_ids,
        img_ids=img_ids,
        attention_mask=attn_mask,
        return_dict=False,
    )[0]
    pred = ChromaPipeline._unpack_latents(pred, height=h * vsf, width=w * vsf, vae_scale_factor=vsf)
    target = noise - x0
    return F.mse_loss(pred.float(), target.float(), reduction="mean")


def save_lora(pipe, lora_state_dict, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    ChromaPipeline.save_lora_weights(
        out_dir, transformer_lora_layers=lora_state_dict, safe_serialization=True
    )
    return os.path.join(out_dir, "pytorch_lora_weights.safetensors")


@torch.no_grad()
def generate(pipe, prompt_cond, negative_cond, height, width, steps, guidance, generator):
    dev = pipe._execution_device
    dt = pipe.transformer.dtype
    kw = dict(
        prompt_embeds=prompt_cond["prompt_embeds"].to(dev, dt),
        prompt_attention_mask=prompt_cond["prompt_attention_mask"].to(dev, dt),
    )
    if guidance > 1:
        if negative_cond is None:
            raise ValueError("Chroma uses true CFG: pass negative_cond (e.g. encode_prompts(pipe, [''])) when guidance > 1")
        kw["negative_prompt_embeds"] = negative_cond["prompt_embeds"].to(dev, dt)
        kw["negative_prompt_attention_mask"] = negative_cond["prompt_attention_mask"].to(dev, dt)
    out = pipe(
        height=height,
        width=width,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=generator,
        output_type="pil",
        **kw,
    )
    return out.images[0]

In [ ]:
%%writefile lora_adapters/longcat_image.py
"""LongCat-Image (meituan-longcat/LongCat-Image) adapter.

diffusers 0.42.1 ships no LongCat training script and LongCatImagePipeline has no LoRA loader mixin.
The reference used here is the official Meituan LoRA script:
  https://github.com/meituan-longcat/LongCat-Image/blob/main/train_examples/lora/train_lora.py
which trains against diffusers' LongCatImageTransformer2DModel:
  latents = (vae_latents - shift_factor) * scaling_factor
  x_t = (1 - s) * x0 + s * noise ; target = noise - x0
  transformer(pack(x_t), prompt_embeds, timestep=s (model multiplies by 1000 internally), img_ids, txt_ids, guidance=None)
  text: 512 padded tokens (prefix/suffix template stripped), no attention mask in the transformer
  txt_ids = text pos ids (modality 0), img_ids = image pos ids (modality 1) starting at (512, 512)
Real checkpoint (transformer/config.json): 10 double + 20 single blocks (NOT the class defaults 19/38), 24 heads x 128,
joint_attention_dim 3584, guidance_embeds false; ~6B params. LORA_TARGET_MODULES match 180 Linear modules there
(120 in transformer_blocks, 60 in single_transformer_blocks), same as the official target list.
"""
import os

import torch
from diffusers import AutoencoderKL, FlowMatchEulerDiscreteScheduler, LongCatImagePipeline, LongCatImageTransformer2DModel
from diffusers.loaders.lora_base import LoraBaseMixin
from diffusers.loaders.lora_pipeline import CogView4LoraLoaderMixin
from diffusers.pipelines.longcat_image.pipeline_longcat_image import prepare_pos_ids, split_quotation


class LongCatImageLoraLoaderMixin(CogView4LoraLoaderMixin):
    """Single-transformer LoRA loader (diffusers has none for LongCat in 0.42.1).

    Also accepts the official Meituan PEFT checkpoints (adapter_model.safetensors with
    'base_model.model.' prefixed keys) by remapping them to 'transformer.'.
    """

    @classmethod
    def lora_state_dict(cls, pretrained_model_name_or_path_or_dict, **kwargs):
        if isinstance(pretrained_model_name_or_path_or_dict, str) and os.path.isdir(pretrained_model_name_or_path_or_dict):
            if "weight_name" not in kwargs and os.path.isfile(
                os.path.join(pretrained_model_name_or_path_or_dict, "adapter_model.safetensors")
            ) and not os.path.isfile(os.path.join(pretrained_model_name_or_path_or_dict, "pytorch_lora_weights.safetensors")):
                kwargs["weight_name"] = "adapter_model.safetensors"
        out = super().lora_state_dict(pretrained_model_name_or_path_or_dict, **kwargs)
        sd, meta = (out if isinstance(out, tuple) else (out, None))
        if any(k.startswith("base_model.model.") for k in sd):
            sd = {"transformer." + k[len("base_model.model."):].replace(".default.", "."): v for k, v in sd.items()}
        return (sd, meta) if isinstance(out, tuple) else sd


class LongCatImageLoraPipeline(LongCatImageLoraLoaderMixin, LongCatImagePipeline):
    """LongCatImagePipeline + LoRA loading + support for prompt_embeds without prompt strings."""

    def encode_prompt(self, prompt=None, num_images_per_prompt=1, prompt_embeds=None):
        if prompt_embeds is not None:
            # upstream does len(prompt) even when prompt is None -> take batch from the embeds instead
            bs, seq_len, _ = prompt_embeds.shape
            prompt_embeds = prompt_embeds.repeat(1, num_images_per_prompt, 1).view(bs * num_images_per_prompt, seq_len, -1)
            text_ids = prepare_pos_ids(modality_id=0, type="text", start=(0, 0), num_token=seq_len).to(self.device)
            return prompt_embeds.to(self.device), text_ids
        return super().encode_prompt(prompt=prompt, num_images_per_prompt=num_images_per_prompt)

    def _encode_prompt_on(self, prompt, device):
        """Copy of LongCatImagePipeline._encode_prompt (diffusers 0.42.1) with input_ids placed on `device`
        (the text encoder's device) instead of self.device, which returns the first non-CPU module and can
        differ from where the text encoder lives."""
        batch_all_tokens = []
        for each_prompt in prompt:
            all_tokens = []
            for clean_prompt_sub, matched in split_quotation(each_prompt):
                if matched:
                    for sub_word in clean_prompt_sub:
                        all_tokens.extend(self.tokenizer(sub_word, add_special_tokens=False)["input_ids"])
                else:
                    all_tokens.extend(self.tokenizer(clean_prompt_sub, add_special_tokens=False)["input_ids"])
            batch_all_tokens.append(all_tokens[: self.tokenizer_max_length])
        tok = self.tokenizer.pad(
            {"input_ids": batch_all_tokens}, max_length=self.tokenizer_max_length, padding="max_length",
            return_attention_mask=True, return_tensors="pt",
        )
        prefix = self.tokenizer(self.prompt_template_encode_prefix, add_special_tokens=False)["input_ids"]
        suffix = self.tokenizer(self.prompt_template_encode_suffix, add_special_tokens=False)["input_ids"]
        bs = tok.input_ids.size(0)
        ids_dt, m_dt = tok.input_ids.dtype, tok.attention_mask.dtype
        input_ids = torch.cat((torch.tensor(prefix, dtype=ids_dt).unsqueeze(0).expand(bs, -1), tok.input_ids,
                               torch.tensor(suffix, dtype=ids_dt).unsqueeze(0).expand(bs, -1)), dim=-1)
        attention_mask = torch.cat((torch.ones(bs, len(prefix), dtype=m_dt), tok.attention_mask,
                                    torch.ones(bs, len(suffix), dtype=m_dt)), dim=-1)
        out = self.text_encoder(input_ids=input_ids.to(device), attention_mask=attention_mask.to(device),
                                output_hidden_states=True)
        return out.hidden_states[-1].detach()[:, len(prefix):-len(suffix), :]


FAMILY = "longcat_image"
PIPELINE_CLASS = LongCatImageLoraPipeline
DENOISER_ATTR = "transformer"
PREDICTION = "flow"
PIXEL_MULTIPLE = 16  # vae_scale_factor 8 * 2x2 packing
DEFAULT_STEPS = 50
DEFAULT_GUIDANCE = 4.5
LORA_TARGET_MODULES = [
    "attn.to_k",
    "attn.to_q",
    "attn.to_v",
    "attn.to_out.0",
    "attn.add_k_proj",
    "attn.add_q_proj",
    "attn.add_v_proj",
    "attn.to_add_out",
    "ff.net.0.proj",
    "ff.net.2",
    "ff_context.net.0.proj",
    "ff_context.net.2",
]


def _try_processor(path, **kw):
    # text_processor is only used by the optional prompt-rewrite step (disabled here). Qwen2VLProcessor needs
    # torchvision for its video/image sub-processors; without torchvision fall back to None.
    try:
        from transformers import Qwen2VLProcessor

        return Qwen2VLProcessor.from_pretrained(path, **kw)
    except Exception:
        return None


def load_pipeline(model_path: str, dtype: torch.dtype):
    sub = {"subfolder": "text_processor"} if not os.path.isdir(model_path) or os.path.isdir(
        os.path.join(model_path, "text_processor")) else {}
    proc = _try_processor(model_path, **sub)
    return PIPELINE_CLASS.from_pretrained(model_path, torch_dtype=dtype, text_processor=proc)


def tiny_pipeline(dtype: torch.dtype):
    from transformers import Qwen2_5_VLConfig, Qwen2_5_VLForConditionalGeneration, Qwen2Tokenizer

    repo = "hf-internal-testing/tiny-random-Qwen2VLForConditionalGeneration"
    torch.manual_seed(0)
    transformer = LongCatImageTransformer2DModel(
        patch_size=1, in_channels=64, num_layers=1, num_single_layers=1, attention_head_dim=16,
        num_attention_heads=2, joint_attention_dim=16, pooled_projection_dim=16, axes_dims_rope=[4, 6, 6],
    )
    torch.manual_seed(0)
    vae = AutoencoderKL(
        in_channels=3, out_channels=3,
        down_block_types=["DownEncoderBlock2D"] * 4, up_block_types=["UpDecoderBlock2D"] * 4,
        block_out_channels=[8, 8, 8, 8], layers_per_block=1, latent_channels=16, norm_num_groups=4,
        use_quant_conv=False, use_post_quant_conv=False, shift_factor=0.1159, scaling_factor=0.3611,
    )
    scheduler = FlowMatchEulerDiscreteScheduler(
        use_dynamic_shifting=True, base_shift=0.5, max_shift=1.15, base_image_seq_len=256, max_image_seq_len=4096,
    )
    torch.manual_seed(0)
    config = Qwen2_5_VLConfig(
        text_config={
            "hidden_size": 16, "intermediate_size": 16, "num_hidden_layers": 2, "num_attention_heads": 2,
            "num_key_value_heads": 2, "max_position_embeddings": 2048,
            "rope_scaling": {"mrope_section": [1, 1, 2], "rope_type": "default", "type": "default"},
            "rope_theta": 1000000.0,
        },
        vision_config={"depth": 2, "hidden_size": 16, "intermediate_size": 16, "num_heads": 2, "out_hidden_size": 16},
        hidden_size=16, vocab_size=152064,
        vision_end_token_id=151653, vision_start_token_id=151652, vision_token_id=151654,
    )
    text_encoder = Qwen2_5_VLForConditionalGeneration(config).eval()
    tokenizer = Qwen2Tokenizer.from_pretrained(repo)
    processor = _try_processor(repo)
    pipe = PIPELINE_CLASS(
        scheduler=scheduler, vae=vae, text_encoder=text_encoder, tokenizer=tokenizer,
        text_processor=processor, transformer=transformer,
    )
    return pipe.to(dtype=dtype)


@torch.no_grad()
def encode_prompts(pipe, prompts, device, dtype):
    pipe.text_encoder.to(device)
    out = []
    for p in prompts:
        # pipeline's own tokenisation: quote-splitting, pad to 512, template prefix/suffix stripped
        embeds = pipe._encode_prompt_on([p], device)
        out.append({"prompt_embeds": embeds.to(dtype).cpu()})
    return out


@torch.no_grad()
def encode_images(pipe, pixel_values, device, dtype):
    vae = pipe.vae
    x = pixel_values.to(device=device, dtype=vae.dtype)
    lat = vae.encode(x).latent_dist.sample()
    lat = (lat - vae.config.shift_factor) * vae.config.scaling_factor
    return lat.float().cpu()


def forward_loss(pipe, denoiser, x0, cond, noise, sigma):
    B, C, h, w = x0.shape
    dtype = next(p for p in denoiser.parameters()).dtype
    device = x0.device
    s = sigma.to(device=device, dtype=x0.dtype).view(-1, 1, 1, 1)
    x_t = (1 - s) * x0 + s * noise
    packed = pipe._pack_latents(x_t.to(dtype), B, C, h, w)

    embeds = cond["prompt_embeds"].to(device=device, dtype=dtype)
    if embeds.shape[0] != B:
        embeds = embeds.expand(B, -1, -1)
    seq = embeds.shape[1]
    img_ids = prepare_pos_ids(modality_id=1, type="image", start=(seq, seq), height=h // 2, width=w // 2).to(device)
    txt_ids = prepare_pos_ids(modality_id=0, type="text", start=(0, 0), num_token=seq).to(device)

    pred = denoiser(
        hidden_states=packed, encoder_hidden_states=embeds,
        timestep=sigma.to(device=device, dtype=dtype),  # in [0,1]; model multiplies by 1000 internally
        img_ids=img_ids, txt_ids=txt_ids, guidance=None, return_dict=False,
    )[0]
    pred = pipe._unpack_latents(pred, h * pipe.vae_scale_factor, w * pipe.vae_scale_factor, pipe.vae_scale_factor)
    target = noise - x0
    return torch.mean((pred.float() - target.float()) ** 2)


def save_lora(pipe, lora_state_dict, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    PIPELINE_CLASS.save_lora_weights(out_dir, transformer_lora_layers=lora_state_dict, safe_serialization=True)
    return os.path.join(out_dir, "pytorch_lora_weights.safetensors")


@torch.no_grad()
def generate(pipe, prompt_cond, negative_cond, height, width, steps, guidance, generator):
    device = pipe._execution_device
    dtype = pipe.transformer.dtype
    pe = prompt_cond["prompt_embeds"].to(device, dtype)
    if negative_cond is not None:
        npe = negative_cond["prompt_embeds"].to(device, dtype)
    elif guidance > 1:
        # The pipeline's own unconditional branch is the encoded empty string (and the official training drops
        # prompts to ''), so zeros would be out of distribution. Text encoder is on CPU here -> require the cache.
        raise ValueError("longcat_image.generate: guidance > 1 needs negative_cond = encode_prompts(pipe, [''], ...)[0]")
    else:
        npe = None
    return pipe(
        prompt_embeds=pe, negative_prompt_embeds=npe, height=height, width=width,
        num_inference_steps=steps, guidance_scale=guidance, generator=generator,
        enable_prompt_rewrite=False, enable_cfg_renorm=True, output_type="pil",
    ).images[0]

In [ ]:
%%writefile lora_adapters/sdxl.py
"""SDXL adapter (StableDiffusionXLPipeline + UNet2DConditionModel).

Mirrors diffusers v0.42.1 examples/text_to_image/train_text_to_image_lora_sdxl.py:
  - latents = vae.encode(px).latent_dist.sample() * vae.config.scaling_factor (VAE in fp32, no shift)
  - noise scheduler = DDPMScheduler built from the model's scheduler config (official uses
    DDPMScheduler.from_pretrained(..., subfolder="scheduler")), x_t = add_noise(x0, noise, t)
  - integer timesteps in [0, num_train_timesteps) fed to the UNet
  - target = noise (epsilon) or get_velocity(x0, noise, t) (v_prediction), per scheduler prediction_type
  - added_cond_kwargs = {"text_embeds": pooled (text_encoder_2), "time_ids": orig_size + crop + target_size}
  - prompt_embeds = concat(penultimate hidden states of both CLIP encoders), no attention mask
  - LoRA on UNet attention: to_k, to_q, to_v, to_out.0
"""
import os

import torch
import torch.nn.functional as F
from diffusers import (
    AutoencoderKL,
    DDPMScheduler,
    EulerDiscreteScheduler,
    StableDiffusionXLPipeline,
    UNet2DConditionModel,
)
from diffusers.utils import convert_state_dict_to_diffusers

FAMILY = "sdxl"
PIPELINE_CLASS = StableDiffusionXLPipeline
DENOISER_ATTR = "unet"
PREDICTION = "eps"  # default; actual target follows pipe.scheduler.config.prediction_type
PIXEL_MULTIPLE = 64
DEFAULT_STEPS = 50  # StableDiffusionXLPipeline.__call__ default (official script validation uses 25)
DEFAULT_GUIDANCE = 5.0  # StableDiffusionXLPipeline.__call__ default
LORA_TARGET_MODULES = ["to_k", "to_q", "to_v", "to_out.0"]


def load_pipeline(model_path: str, dtype: torch.dtype):
    pipe = StableDiffusionXLPipeline.from_pretrained(model_path, torch_dtype=dtype)
    return pipe


def tiny_pipeline(dtype: torch.dtype):
    # copied from tests/pipelines/stable_diffusion_xl/test_stable_diffusion_xl.py::get_dummy_components
    from transformers import CLIPTextConfig, CLIPTextModel, CLIPTextModelWithProjection, CLIPTokenizer

    torch.manual_seed(0)
    unet = UNet2DConditionModel(
        block_out_channels=(2, 4),
        layers_per_block=2,
        sample_size=32,
        in_channels=4,
        out_channels=4,
        down_block_types=("DownBlock2D", "CrossAttnDownBlock2D"),
        up_block_types=("CrossAttnUpBlock2D", "UpBlock2D"),
        attention_head_dim=(2, 4),
        use_linear_projection=True,
        addition_embed_type="text_time",
        addition_time_embed_dim=8,
        transformer_layers_per_block=(1, 2),
        projection_class_embeddings_input_dim=80,  # 6 * 8 + 32
        cross_attention_dim=64,
        norm_num_groups=1,
    )
    scheduler = EulerDiscreteScheduler(
        beta_start=0.00085,
        beta_end=0.012,
        steps_offset=1,
        beta_schedule="scaled_linear",
        timestep_spacing="leading",
    )
    torch.manual_seed(0)
    vae = AutoencoderKL(
        block_out_channels=[32, 64],
        in_channels=3,
        out_channels=3,
        down_block_types=["DownEncoderBlock2D", "DownEncoderBlock2D"],
        up_block_types=["UpDecoderBlock2D", "UpDecoderBlock2D"],
        latent_channels=4,
        sample_size=128,
    )
    torch.manual_seed(0)
    te_cfg = CLIPTextConfig(
        bos_token_id=0,
        eos_token_id=2,
        hidden_size=32,
        intermediate_size=37,
        layer_norm_eps=1e-05,
        num_attention_heads=4,
        num_hidden_layers=5,
        pad_token_id=1,
        vocab_size=1000,
        hidden_act="gelu",
        projection_dim=32,
    )
    text_encoder = CLIPTextModel(te_cfg)
    tokenizer = CLIPTokenizer.from_pretrained("hf-internal-testing/tiny-random-clip")
    text_encoder_2 = CLIPTextModelWithProjection(te_cfg)
    tokenizer_2 = CLIPTokenizer.from_pretrained("hf-internal-testing/tiny-random-clip")
    pipe = StableDiffusionXLPipeline(
        vae=vae,
        text_encoder=text_encoder,
        text_encoder_2=text_encoder_2,
        tokenizer=tokenizer,
        tokenizer_2=tokenizer_2,
        unet=unet,
        scheduler=scheduler,
        image_encoder=None,
        feature_extractor=None,
    )
    return pipe.to(dtype=dtype)


@torch.no_grad()
def encode_prompts(pipe, prompts, device, dtype):
    out = []
    for p in prompts:
        pe, _, pooled, _ = pipe.encode_prompt(
            prompt=p,
            prompt_2=p,
            device=device,
            num_images_per_prompt=1,
            do_classifier_free_guidance=False,
        )
        out.append({
            "prompt_embeds": pe.to("cpu", dtype),  # (1, 77, 2048) penultimate layers concat
            "pooled_prompt_embeds": pooled.to("cpu", dtype),  # (1, 1280) text_encoder_2 projection
        })
    return out


@torch.no_grad()
def encode_images(pipe, pixel_values, device, dtype):
    vae = pipe.vae
    orig_dtype = vae.dtype
    # official script keeps the SDXL VAE in fp32 (fp16/bf16 NaN issue) unless a fixed VAE is supplied
    if orig_dtype != torch.float32:
        vae.to(torch.float32)
    try:
        px = pixel_values.to(device=device, dtype=torch.float32)
        lat = vae.encode(px).latent_dist.sample()
        lat = lat * vae.config.scaling_factor
    finally:
        if orig_dtype != torch.float32:
            vae.to(orig_dtype)
    return lat.float().cpu()


def _train_scheduler(pipe):
    sch = getattr(pipe, "_lvw_ddpm", None)
    if sch is None or sch.config.prediction_type != pipe.scheduler.config.prediction_type:
        sch = DDPMScheduler.from_config(pipe.scheduler.config)
        pipe._lvw_ddpm = sch
    return sch


def _time_ids(pipe, x0, device, dtype, cond=None):
    # original_size + crops_coords_top_left + target_size, in pixels.
    # Official script (train_dreambooth_lora_sdxl.py) uses each image's real original size, its random-crop
    # top-left and target_size=(resolution, resolution). The notebook may pass these per sample via optional
    # cond keys "original_size" (h, w) and "crop_top_left" (top, left); otherwise default to the bucketed
    # training size with no crop, which matches how inference sets height/width for aspect-ratio buckets.
    vsf = pipe.vae_scale_factor
    h, w = x0.shape[-2] * vsf, x0.shape[-1] * vsf
    cond = cond or {}
    orig = cond.get("original_size", (h, w))
    crop = cond.get("crop_top_left", (0, 0))
    def _pair(v):
        v = torch.as_tensor(v, dtype=torch.float32).reshape(-1, 2)
        return v.expand(x0.shape[0], 2) if v.shape[0] == 1 else v
    tgt = torch.tensor([[h, w]], dtype=torch.float32).expand(x0.shape[0], 2)
    ids = torch.cat([_pair(orig), _pair(crop), tgt], dim=1)
    return ids.to(device=device, dtype=dtype)


def forward_loss(pipe, denoiser, x0, cond, noise, sigma):
    sch = _train_scheduler(pipe)
    device = x0.device
    wdtype = next(p for p in denoiser.parameters()).dtype
    T = sch.config.num_train_timesteps
    t = (sigma.float() * (T - 1)).round().long().clamp(0, T - 1).to(device)
    if t.shape[0] != x0.shape[0]:
        t = t.expand(x0.shape[0])
    x0f, nf = x0.float(), noise.float()
    x_t = sch.add_noise(x0f, nf, t)
    pe = cond["prompt_embeds"].to(device, wdtype)
    pooled = cond["pooled_prompt_embeds"].to(device, wdtype)
    if pe.shape[0] != x0.shape[0]:
        pe = pe.expand(x0.shape[0], -1, -1)
        pooled = pooled.expand(x0.shape[0], -1)
    added = {"text_embeds": pooled, "time_ids": _time_ids(pipe, x0, device, wdtype, cond)}
    pred = denoiser(x_t.to(wdtype), t, pe, added_cond_kwargs=added, return_dict=False)[0]
    ptype = sch.config.prediction_type
    if ptype == "epsilon":
        target = nf
    elif ptype == "v_prediction":
        target = sch.get_velocity(x0f, nf, t)
    else:
        raise ValueError(f"Unknown prediction type {ptype}")
    return F.mse_loss(pred.float(), target.float(), reduction="mean")


def save_lora(pipe, lora_state_dict, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    StableDiffusionXLPipeline.save_lora_weights(
        save_directory=out_dir,
        unet_lora_layers=convert_state_dict_to_diffusers(lora_state_dict),
        safe_serialization=True,
    )
    return os.path.join(out_dir, "pytorch_lora_weights.safetensors")


@torch.no_grad()
def generate(pipe, prompt_cond, negative_cond, height, width, steps, guidance, generator):
    dev = pipe._execution_device
    dt = pipe.unet.dtype
    pe = prompt_cond["prompt_embeds"].to(dev, dt)
    pooled = prompt_cond["pooled_prompt_embeds"].to(dev, dt)
    if negative_cond is not None:
        npe = negative_cond["prompt_embeds"].to(dev, dt)
        npooled = negative_cond["pooled_prompt_embeds"].to(dev, dt)
    else:
        # force_zeros_for_empty_prompt behaviour of SDXL (no text encoder call needed)
        npe, npooled = torch.zeros_like(pe), torch.zeros_like(pooled)
    if npe.shape[1] != pe.shape[1]:
        L = max(npe.shape[1], pe.shape[1])
        pe = F.pad(pe, (0, 0, 0, L - pe.shape[1]))
        npe = F.pad(npe, (0, 0, 0, L - npe.shape[1]))
    return pipe(
        prompt_embeds=pe,
        pooled_prompt_embeds=pooled,
        negative_prompt_embeds=npe,
        negative_pooled_prompt_embeds=npooled,
        height=height,
        width=width,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=generator,
        output_type="pil",
    ).images[0]

In [ ]:
%%writefile lora_adapters/sd3.py
"""Stable Diffusion 3 / 3.5 adapter (StableDiffusion3Pipeline + SD3Transformer2DModel, rectified flow).

Mirrors diffusers v0.42.1 examples/dreambooth/train_dreambooth_lora_sd3.py:
  * latents  = (vae.encode(x).latent_dist.sample() - shift_factor) * scaling_factor
  * x_t      = (1 - sigma) * x0 + sigma * noise
  * timestep = sigma * num_train_timesteps (scheduler.timesteps units, 0..1000), passed as-is
  * no packing, no ids, no attention mask, no guidance embedding (SD3 transformer patchifies internally)
  * text: CLIP-L + CLIP-G penultimate hidden states (concat, zero-padded to T5 width) ++ T5 (max_sequence_length=77
    in the training script), pooled = concat of both CLIP pooled projections
  * loss: official default --precondition_outputs=1 -> pred_x0 = model_pred * (-sigma) + x_t, target = x0.
    Set PRECONDITION_OUTPUTS = False to use target = noise - x0 directly (the script's --precondition_outputs=0).
"""
import os
import warnings

import torch
from diffusers import (
    AutoencoderKL,
    FlowMatchEulerDiscreteScheduler,
    SD3Transformer2DModel,
    StableDiffusion3Pipeline,
)

FAMILY = "sd3"
PIPELINE_CLASS = StableDiffusion3Pipeline
DENOISER_ATTR = "transformer"
PREDICTION = "flow"
PIXEL_MULTIPLE = 16  # vae_scale_factor 8 * transformer patch_size 2
DEFAULT_STEPS = 28  # StableDiffusion3Pipeline.__call__ default
DEFAULT_GUIDANCE = 7.0  # StableDiffusion3Pipeline.__call__ default
LORA_TARGET_MODULES = [
    "attn.add_k_proj",
    "attn.add_q_proj",
    "attn.add_v_proj",
    "attn.to_add_out",
    "attn.to_k",
    "attn.to_out.0",
    "attn.to_q",
    "attn.to_v",
]
MAX_SEQUENCE_LENGTH = 77  # official training script default (pipeline inference default is 256)
PRECONDITION_OUTPUTS = True  # official training script default (--precondition_outputs=1)


def load_pipeline(model_path: str, dtype: torch.dtype):
    # works for a local diffusers-format directory (model_index.json) or a HF repo id
    return StableDiffusion3Pipeline.from_pretrained(model_path, torch_dtype=dtype)


def tiny_pipeline(dtype: torch.dtype):
    # same as diffusers tests/pipelines/stable_diffusion_3 get_dummy_components()
    from transformers import AutoConfig, AutoTokenizer, CLIPTextConfig, CLIPTextModelWithProjection, CLIPTokenizer, T5EncoderModel

    torch.manual_seed(0)
    transformer = SD3Transformer2DModel(
        sample_size=32, patch_size=1, in_channels=4, num_layers=1, attention_head_dim=8, num_attention_heads=4,
        caption_projection_dim=32, joint_attention_dim=32, pooled_projection_dim=64, out_channels=4,
    )
    clip_cfg = CLIPTextConfig(
        bos_token_id=0, eos_token_id=2, hidden_size=32, intermediate_size=37, layer_norm_eps=1e-05,
        num_attention_heads=4, num_hidden_layers=5, pad_token_id=1, vocab_size=1000, hidden_act="gelu",
        projection_dim=32,
    )
    torch.manual_seed(0)
    text_encoder = CLIPTextModelWithProjection(clip_cfg)
    torch.manual_seed(0)
    text_encoder_2 = CLIPTextModelWithProjection(clip_cfg)
    torch.manual_seed(0)
    text_encoder_3 = T5EncoderModel(AutoConfig.from_pretrained("hf-internal-testing/tiny-random-t5"))
    tokenizer = CLIPTokenizer.from_pretrained("hf-internal-testing/tiny-random-clip")
    tokenizer_2 = CLIPTokenizer.from_pretrained("hf-internal-testing/tiny-random-clip")
    tokenizer_3 = AutoTokenizer.from_pretrained("hf-internal-testing/tiny-random-t5")
    torch.manual_seed(0)
    vae = AutoencoderKL(
        sample_size=32, in_channels=3, out_channels=3, block_out_channels=(4,), layers_per_block=1,
        latent_channels=4, norm_num_groups=1, use_quant_conv=False, use_post_quant_conv=False,
        shift_factor=0.0609, scaling_factor=1.5035,
    )
    pipe = StableDiffusion3Pipeline(
        scheduler=FlowMatchEulerDiscreteScheduler(), vae=vae, text_encoder=text_encoder, tokenizer=tokenizer,
        text_encoder_2=text_encoder_2, tokenizer_2=tokenizer_2, text_encoder_3=text_encoder_3,
        tokenizer_3=tokenizer_3, transformer=transformer, image_encoder=None, feature_extractor=None,
    )
    return pipe.to(dtype=dtype)


@torch.no_grad()
def encode_prompts(pipe, prompts, device, dtype):
    out = []
    for p in prompts:
        # pipe.encode_prompt == the training script's encode_prompt (clip penultimate layer, zero-pad, concat T5);
        # if text_encoder_3 is None the pipeline substitutes zeros for T5, like the official pipeline.
        pe, _, pooled, _ = pipe.encode_prompt(
            prompt=p, prompt_2=None, prompt_3=None, device=device, num_images_per_prompt=1,
            do_classifier_free_guidance=False, max_sequence_length=MAX_SEQUENCE_LENGTH,
        )
        out.append({
            "prompt_embeds": pe.to("cpu", dtype),
            "pooled_prompt_embeds": pooled.to("cpu", dtype),
        })
    return out


@torch.no_grad()
def encode_images(pipe, pixel_values, device, dtype):
    # official script keeps the VAE in fp32 for latent encoding (train_dreambooth_lora_sd3.py:1237)
    vae = pipe.vae
    orig_dtype = vae.dtype
    vae.to(dtype=torch.float32)
    try:
        lat = vae.encode(pixel_values.to(device, torch.float32)).latent_dist.sample()
        lat = (lat - vae.config.shift_factor) * vae.config.scaling_factor
    finally:
        vae.to(dtype=orig_dtype)
    return lat.float().cpu()


def forward_loss(pipe, denoiser, x0, cond, noise, sigma):
    wdt = next(denoiser.parameters()).dtype
    x0 = x0.to(wdt)
    noise = noise.to(wdt)
    s = sigma.to(x0.device, wdt).view(-1, *([1] * (x0.ndim - 1)))
    x_t = (1.0 - s) * x0 + s * noise
    timesteps = sigma.to(x0.device, torch.float32) * pipe.scheduler.config.num_train_timesteps
    pe = cond["prompt_embeds"].to(x0.device, wdt)
    pooled = cond["pooled_prompt_embeds"].to(x0.device, wdt)
    if pe.shape[0] != x0.shape[0]:
        pe = pe.expand(x0.shape[0], -1, -1)
        pooled = pooled.expand(x0.shape[0], -1)
    model_pred = denoiser(
        hidden_states=x_t, timestep=timesteps, encoder_hidden_states=pe, pooled_projections=pooled,
        return_dict=False,
    )[0]
    if PRECONDITION_OUTPUTS:
        model_pred = model_pred * (-s) + x_t
        target = x0
    else:
        target = noise - x0
    return torch.mean((model_pred.float() - target.float()) ** 2)


def save_lora(pipe, lora_state_dict, out_dir):
    # official script passes get_peft_model_state_dict() output unchanged (peft lora_A/lora_B keys)
    os.makedirs(out_dir, exist_ok=True)
    StableDiffusion3Pipeline.save_lora_weights(
        out_dir, transformer_lora_layers=lora_state_dict
    )
    return os.path.join(out_dir, "pytorch_lora_weights.safetensors")


@torch.no_grad()
def generate(pipe, prompt_cond, negative_cond, height, width, steps, guidance, generator):
    dev = pipe._execution_device
    dt = pipe.transformer.dtype
    kw = dict(
        prompt_embeds=prompt_cond["prompt_embeds"].to(dev, dt),
        pooled_prompt_embeds=prompt_cond["pooled_prompt_embeds"].to(dev, dt),
    )
    if negative_cond is not None:
        kw["negative_prompt_embeds"] = negative_cond["prompt_embeds"].to(dev, dt)
        kw["negative_pooled_prompt_embeds"] = negative_cond["pooled_prompt_embeds"].to(dev, dt)
    elif guidance > 1.0:
        # Official validation runs CFG with the pipeline's empty negative prompt. Encode "" once with the text
        # encoders on whatever device they currently sit on (CPU during training), cache it on the pipe, move to dev.
        neg = getattr(pipe, "_lvw_empty_negative_cond", None)
        if neg is None and pipe.text_encoder is not None:
            te_dev = next(pipe.text_encoder.parameters()).device
            neg = encode_prompts(pipe, [""], te_dev, dt)[0]
            pipe._lvw_empty_negative_cond = neg
        if neg is not None:
            kw["negative_prompt_embeds"] = neg["prompt_embeds"].to(dev, dt)
            kw["negative_pooled_prompt_embeds"] = neg["pooled_prompt_embeds"].to(dev, dt)
        else:
            warnings.warn("sd3.generate: no negative_cond and no text encoder; CFG disabled (guidance=1.0)")
            guidance = 1.0
    return pipe(
        height=height, width=width, num_inference_steps=steps, guidance_scale=guidance, generator=generator,
        max_sequence_length=MAX_SEQUENCE_LENGTH, **kw,
    ).images[0]

In [ ]:
ADAPTER_DIR = Path("lora_adapters").resolve()
if str(ADAPTER_DIR) not in sys.path:
    sys.path.insert(0, str(ADAPTER_DIR))
SUPPORTED = sorted(p.stem for p in ADAPTER_DIR.glob("*.py"))
if USER["MODEL_FAMILY"] not in SUPPORTED:
    raise ValueError(f"MODEL_FAMILY='{USER['MODEL_FAMILY']}' 는 지원하지 않습니다. 고를 수 있는 값: {SUPPORTED}")
A = importlib.import_module(USER["MODEL_FAMILY"])
for _name in ["PIPELINE_CLASS", "DENOISER_ATTR", "PREDICTION", "PIXEL_MULTIPLE", "DEFAULT_STEPS", "DEFAULT_GUIDANCE",
              "LORA_TARGET_MODULES", "load_pipeline", "tiny_pipeline", "encode_prompts", "encode_images",
              "forward_loss", "save_lora", "generate"]:
    if not hasattr(A, _name):
        raise AttributeError(f"어댑터 {USER['MODEL_FAMILY']} 에 {_name} 가 없습니다")
print("어댑터:", A.FAMILY, "| 파이프라인:", A.PIPELINE_CLASS.__name__, "| 예측:", A.PREDICTION,
      "| 픽셀 배수:", A.PIXEL_MULTIPLE, "| LoRA 대상:", A.LORA_TARGET_MODULES)

### 0-4. 장치·정밀도 결정, 시드 고정, 모델 불러오기

In [ ]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(FIXED["seed"])
if TEST_FAMILY:
    DEVICE = torch.device("cuda" if TEST_DEVICE == "cuda" and torch.cuda.is_available() else "cpu")
else:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if FIXED["mixed_precision"] == "bf16" and DEVICE.type == "cuda" and torch.cuda.is_bf16_supported():
    WEIGHT_DTYPE = torch.bfloat16
elif FIXED["mixed_precision"] in ("bf16", "fp16") and DEVICE.type == "cuda":
    WEIGHT_DTYPE = torch.float16     # bf16 을 못 쓰는 GPU
else:
    WEIGHT_DTYPE = torch.float32     # CPU 는 fp32
print("장치:", DEVICE, "| 가중치 dtype:", WEIGHT_DTYPE)

def check_model_path(mp):
    """로컬 폴더면 diffusers 형식과 모델 계열을 확인하고, 아니면 Hugging Face 저장소 이름인지 확인한다."""
    p = Path(mp).expanduser()
    if p.exists():
        if p.is_file():
            raise ValueError(f"MODEL_PATH 가 파일입니다: {mp}. ComfyUI 용 단일 .safetensors 는 쓸 수 없고, "
                             "model_index.json 이 있는 diffusers 형식 폴더가 필요합니다.")
        mi = p / "model_index.json"
        if not mi.exists():
            raise FileNotFoundError(f"{mp} 에 model_index.json 이 없습니다. diffusers 형식 폴더가 필요합니다.")
        cls_name = json.loads(mi.read_text(encoding="utf-8")).get("_class_name", "")
        ok = {c.__name__ for c in A.PIPELINE_CLASS.__mro__ if issubclass(c, diffusers.DiffusionPipeline)}
        if cls_name and cls_name not in ok:
            raise ValueError(f"이 폴더는 {cls_name} 모델입니다. MODEL_FAMILY='{A.FAMILY}' 는 "
                             f"{A.PIPELINE_CLASS.__name__} 용입니다. MODEL_FAMILY 를 확인하세요.")
        return str(p)
    if p.is_absolute() or mp.startswith((".", "~")) or not re.fullmatch(r"[\w.-]+/[\w.-]+", mp):
        raise FileNotFoundError(f"MODEL_PATH 가 없습니다: {mp}")
    # Hugging Face 저장소 이름 (예: Qwen/Qwen-Image-2512): 큰 파일을 받기 전에 model_index.json 만 받아 계열 확인
    try:
        from huggingface_hub import hf_hub_download
        cls_name = json.loads(Path(hf_hub_download(mp, "model_index.json")).read_text(encoding="utf-8")).get("_class_name", "")
    except Exception as e:
        print(f"참고: {mp} 의 model_index.json 을 받지 못해 계열 확인을 건너뜁니다 ({type(e).__name__})")
        return mp
    ok = {c.__name__ for c in A.PIPELINE_CLASS.__mro__ if issubclass(c, diffusers.DiffusionPipeline)}
    if cls_name and cls_name not in ok:
        raise ValueError(f"{mp} 는 {cls_name} 모델입니다. MODEL_FAMILY='{A.FAMILY}' 는 {A.PIPELINE_CLASS.__name__} 용입니다.")
    return mp

t0 = time.time()
if TEST_FAMILY:
    pipe = A.tiny_pipeline(WEIGHT_DTYPE)
else:
    pipe = A.load_pipeline(check_model_path(USER["MODEL_PATH"]), WEIGHT_DTYPE)
if not isinstance(pipe, A.PIPELINE_CLASS):
    raise TypeError(f"불러온 파이프라인 {type(pipe).__name__} 이 {A.PIPELINE_CLASS.__name__} 가 아닙니다. MODEL_FAMILY 를 확인하세요.")
denoiser = getattr(pipe, A.DENOISER_ATTR)
for _c in pipe.components.values():
    if isinstance(_c, torch.nn.Module):
        _c.requires_grad_(False)
        _c.eval()
_param_rows = [(k, type(v).__name__, sum(p.numel() for p in v.parameters()) / 1e6)
               for k, v in pipe.components.items() if isinstance(v, torch.nn.Module)]
display_df = pd.DataFrame(_param_rows, columns=["구성요소", "클래스", "파라미터(백만)"])
print(display_df.to_string(index=False))
print(f"불러오기 {time.time() - t0:.1f}초")

## 1. EDA (데이터 살펴보기)

In [ ]:
IMG_EXT = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}

def make_test_dataset(root, n=6):
    """시험 실행용 가짜 데이터셋: 서로 다른 비율의 무작위 색 이미지 + 캡션"""
    root.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(0)
    sizes = [(64, 96), (96, 64), (80, 80), (64, 112), (112, 64), (72, 72)]
    for i in range(n):
        w, h = sizes[i % len(sizes)]
        arr = (rng.random((h, w, 3)) * 255).astype("uint8")
        Image.fromarray(arr).save(root / f"img_{i:02d}.png")
        if i != n - 1:   # 마지막 하나는 캡션 없음 → 경고 경로 시험
            (root / f"img_{i:02d}.txt").write_text(f"{USER['TRIGGER']}, test image {i}", encoding="utf-8")
    Image.fromarray(arr).save(root / "dup_00.png")   # 중복 → 제외 경로 시험
    # 문제 사례: EXIF 회전(저장은 가로, 보이는 것은 세로), 16비트 흑백, 여러 캡션 인코딩
    exif = Image.Exif(); exif[0x0112] = 6
    Image.fromarray((rng.random((64, 96, 3)) * 255).astype("uint8")).save(root / "rot_exif.jpg", exif=exif.tobytes())
    (root / "rot_exif.txt").write_bytes(f"{USER['TRIGGER']}, rotated".encode("utf-8-sig"))          # BOM
    Image.fromarray((rng.random((80, 64)) * 65535).astype("uint16")).save(root / "gray16.png")
    (root / "gray16.txt").write_bytes(f"({USER['TRIGGER']}) 회색 이미지".encode("cp949"))              # CP949
    Image.fromarray((rng.random((72, 88, 3)) * 255).astype("uint8")).save(root / "utf16.png")
    (root / "utf16.txt").write_bytes(f"{USER['TRIGGER']}, utf16 caption".encode("utf-16"))           # UTF-16
    Image.fromarray(rng.random((70, 90)).astype("float32"), "F").save(root / "float01.tif")           # 0~1 실수
    (root / "float01.txt").write_bytes(f"{USER['TRIGGER']}가 서 있음".encode("utf-16-le"))             # BOM 없는 UTF-16
    rgba = (rng.random((66, 84, 4)) * 255).astype("uint8"); rgba[..., 3] = 0
    Image.fromarray(rgba, "RGBA").save(root / "clear.png")                                           # 완전 투명
    (root / "clear.txt").write_text(f"{USER['TRIGGER']}_x, transparent", encoding="utf-8")           # 트리거 아님
    return root

DATA_DIR = make_test_dataset(Path(tempfile.mkdtemp()) / "data") if TEST_FAMILY else Path(USER["DATASET_DIR"])
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"DATASET_DIR 가 없습니다: {DATA_DIR}")

def to_8bit(im):
    """16비트 흑백(I;16)·정수(I)·실수(F) 이미지를 0~255 로 줄인다. 그대로 RGB 로 바꾸면 255 에서 잘려 하얗게 된다."""
    if im.mode.startswith("I;16"):
        a = np.asarray(im, dtype=np.float32) / 257.0
    elif im.mode in ("I", "F"):
        a = np.asarray(im, dtype=np.float32)
        hi = float(np.nanmax(a)) if a.size else 0.0
        if im.mode == "F" and hi <= 1.0:
            a = a * 255.0                  # 0~1 실수 이미지
        elif hi > 255.0:
            a = a / hi * 255.0             # 범위를 모르는 큰 값은 최댓값 기준
    else:
        return im
    return Image.fromarray(np.nan_to_num(a).clip(0, 255).astype(np.uint8), "L")

def flatten_alpha(im):
    """투명한 부분을 흰 배경에 합성한 RGB. EDA 와 학습이 같은 이미지를 보게 한다."""
    if im.mode in ("RGBA", "LA", "PA") or "transparency" in im.info:
        im = Image.alpha_composite(Image.new("RGBA", im.size, (255, 255, 255, 255)), im.convert("RGBA"))
    return im.convert("RGB")

def read_caption(path):
    """UTF-8(BOM 포함)·UTF-16·CP949 캡션을 읽는다. 모르는 인코딩은 글자를 바꾸고 문제 목록에 남긴다."""
    b = path.read_bytes()
    if b.startswith((b"\xff\xfe", b"\xfe\xff")):
        return b.decode("utf-16").strip()
    if len(b) >= 4 and b"\x00" in b:          # BOM 없는 UTF-16: 한쪽 바이트가 대부분 0
        even, odd = b[0::2], b[1::2]
        if odd.count(0) > len(odd) * 0.6:
            return b.decode("utf-16-le", errors="replace").strip()
        if even.count(0) > len(even) * 0.6:
            return b.decode("utf-16-be", errors="replace").strip()
    try:
        return b.decode("utf-8-sig").strip()
    except UnicodeDecodeError:
        pass
    try:
        text = b.decode("cp949").strip()
        problems.append((path.name, "캡션을 CP949 로 읽음 (내용 확인 권장, UTF-8 로 저장 권장)"))
        return text
    except UnicodeDecodeError:
        pass
    problems.append((path.name, "캡션 인코딩을 알 수 없어 일부 글자를 바꿈 (UTF-8 로 저장 권장)"))
    return b.decode("utf-8", errors="replace").strip()

# 왼쪽은 낱말 경계, 오른쪽은 영문·숫자·밑줄만 막는다 (한글 트리거 뒤에 조사가 붙은 '니키가' 도 트리거로 인정)
TRIGGER_RE = re.compile(r"(?<!\w)" + re.escape(USER["TRIGGER"]) + r"(?![A-Za-z0-9_])")

def blur_score(gray):
    """라플라시안 분산: 낮을수록 흐림"""
    g = gray.astype(np.float32)
    lap = g[1:-1, 1:-1] * -4 + g[:-2, 1:-1] + g[2:, 1:-1] + g[1:-1, :-2] + g[1:-1, 2:]
    return float(lap.var())

rows, problems = [], []
for p in sorted(DATA_DIR.iterdir()):
    if p.suffix.lower() not in IMG_EXT:
        continue
    try:
        with Image.open(p) as im:
            im.load()
            mode, info = im.mode, dict(im.info)
            im_t = to_8bit(ImageOps.exif_transpose(im))      # EXIF 회전 뒤 크기로 묶음을 고른다
            w, h = im_t.size
            rgb = np.asarray(flatten_alpha(im_t))
    except Exception as e:
        problems.append((p.name, f"열 수 없음: {e}")); continue
    if mode.startswith("I;16") or mode in ("I", "F"):
        problems.append((p.name, f"{mode} 이미지를 8비트로 줄여 사용"))
    cap_path = p.with_suffix(".txt")
    caption = read_caption(cap_path) if cap_path.exists() else ""
    rows.append(dict(file=p.name, width=w, height=h, aspect=round(w / h, 3), mode=mode,
                     has_alpha=mode in ("RGBA", "LA", "PA") or "transparency" in info,
                     caption=caption, caption_words=len(caption.split()), has_caption=bool(caption),
                     has_trigger=bool(TRIGGER_RE.search(caption)),
                     sha1=hashlib.sha1(rgb.tobytes()).hexdigest(), blur=blur_score(rgb.mean(axis=2)),
                     brightness=float(rgb.mean())))
if not rows:
    raise RuntimeError(f"{DATA_DIR} 에 이미지가 없습니다 ({sorted(IMG_EXT)})")
df = pd.DataFrame(rows)

# 문제 처리: 중복은 빼고, 작은 이미지는 경고, 캡션·트리거가 없으면 채운다
dup = df.duplicated("sha1", keep="first")
for f in df.loc[dup, "file"]:
    problems.append((f, "같은 이미지가 이미 있어 제외"))
df = df.loc[~dup].reset_index(drop=True)
for _, r in df.iterrows():
    if min(r.width, r.height) < FIXED["min_image_side"]:
        problems.append((r.file, f"짧은 변 {min(r.width, r.height)}px < {FIXED['min_image_side']}px (업스케일 권장)"))
    if not r.has_caption:
        problems.append((r.file, "캡션 파일 없음 → 트리거만으로 학습"))
    elif not r.has_trigger:
        problems.append((r.file, "캡션에 트리거 없음 → 앞에 붙임"))
df["train_caption"] = [c if t else (f"{USER['TRIGGER']}, {c}" if c else USER["TRIGGER"])
                       for c, t in zip(df.caption, df.has_trigger)]
low_blur = df.blur < df.blur.median() * 0.25
for f in df.loc[low_blur, "file"]:
    problems.append((f, "다른 이미지보다 많이 흐림 (확인 권장)"))

if TEST_FAMILY:   # 문제 사례가 바르게 처리됐는지 확인
    _r = df.set_index("file")
    assert _r.loc["rot_exif.jpg", "height"] > _r.loc["rot_exif.jpg", "width"], "EXIF 회전 미반영"
    assert all(_r.loc[f, "has_trigger"] for f in ["rot_exif.jpg", "gray16.png", "utf16.png"]), "트리거 판정 오류"
    assert not _r.loc["rot_exif.jpg", "train_caption"].startswith(USER["TRIGGER"] + ", " + USER["TRIGGER"])
    assert "회색" in _r.loc["gray16.png", "caption"], "CP949 캡션 해석 실패"
    assert 60 < _r.loc["gray16.png", "brightness"] < 200, "16비트 이미지가 하얗게 잘림"
    assert int(dup.sum()) == 1
    assert _r.loc["float01.tif", "has_trigger"] and "\x00" not in _r.loc["float01.tif", "caption"], "BOM 없는 UTF-16 오류"
    assert 60 < _r.loc["float01.tif", "brightness"] < 200, "0~1 실수 이미지가 검게 나옴"
    assert _r.loc["clear.png", "brightness"] > 250, "투명 이미지가 흰 배경에 합성되지 않음"
    assert not _r.loc["clear.png", "has_trigger"], "nkgreen_x 를 트리거로 잘못 인정"
print(f"이미지 {len(df)}장 (제외 {int(dup.sum())}장)")
print(df[["width", "height", "aspect", "caption_words", "blur", "brightness"]].describe().round(2).to_string())
print("\n이미지 모드:", dict(Counter(df["mode"])), "| 투명도 있음:", int(df.has_alpha.sum()))
print("\n문제 목록:" if problems else "\n문제 없음")
for f, msg in problems:
    print(f"  - {f}: {msg}")
if len(df) < 5 and not TEST_FAMILY:
    print("경고: 이미지가 5장보다 적습니다. 계획서 기준은 20~30장입니다.")

## 2. 전처리

### 2-1. 비율별 묶음(bucket) 정하기와 자르기

In [ ]:
M = A.PIXEL_MULTIPLE
TARGET = (2 * M) if TEST_FAMILY else FIXED["resolution"]   # 시험 실행은 아주 작게

def make_buckets(target, aspects, multiple):
    """넓이가 target^2 에 가깝고 multiple 의 배수인 (w,h) 목록"""
    out = set()
    for a in aspects:
        w = max(multiple, int(round(target * math.sqrt(a) / multiple)) * multiple)
        h = max(multiple, int(round(target / math.sqrt(a) / multiple)) * multiple)
        out.add((w, h))
    return sorted(out, key=lambda wh: wh[0] / wh[1])

BUCKETS = make_buckets(TARGET, FIXED["bucket_aspects"], M)

def pick_bucket(w, h):
    a = w / h
    return min(BUCKETS, key=lambda b: abs(math.log(b[0] / b[1]) - math.log(a)))

def load_rgb(path):
    with Image.open(path) as im:
        return flatten_alpha(to_8bit(ImageOps.exif_transpose(im)))

def resize_crop(im, bw, bh):
    """비율을 지키며 덮도록 키운 뒤 가운데를 자름"""
    scale = max(bw / im.width, bh / im.height)
    nw, nh = max(bw, round(im.width * scale)), max(bh, round(im.height * scale))
    im = im.resize((nw, nh), Image.LANCZOS)
    left, top = (nw - bw) // 2, (nh - bh) // 2
    return im.crop((left, top, left + bw, top + bh))

def to_tensor(im):
    arr = np.asarray(im, dtype=np.float32) / 127.5 - 1.0
    return torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).contiguous()

df["bucket"] = [pick_bucket(w, h) for w, h in zip(df.width, df.height)]
processed = [resize_crop(load_rgb(DATA_DIR / f), *b) for f, b in zip(df.file, df.bucket)]
for im, b in zip(processed, df.bucket):
    assert im.size == b and b[0] % M == 0 and b[1] % M == 0, (im.size, b)
print("묶음 후보:", BUCKETS)
print("사용된 묶음:", dict(Counter(df.bucket)))

### 2-2. 잠재값과 텍스트 임베딩 미리 계산 (캐시)

텍스트 인코더와 VAE 는 한 번만 쓰고 GPU 에서 내려, 학습 중에는 디노이저만 GPU 에 둔다.

In [ ]:
def offload(names, device):
    for n in names:
        m = getattr(pipe, n, None)
        if isinstance(m, torch.nn.Module):
            m.to(device)

DENOISER_NAME = A.DENOISER_ATTR
OTHER_MODULES = [k for k, v in pipe.components.items() if isinstance(v, torch.nn.Module) and k != DENOISER_NAME]
offload(OTHER_MODULES, DEVICE)

with torch.no_grad():
    caption_conds = A.encode_prompts(pipe, list(df.train_caption), DEVICE, WEIGHT_DTYPE)
    val_conds = A.encode_prompts(pipe, VALIDATION_PROMPTS, DEVICE, WEIGHT_DTYPE)
    neg_cond = A.encode_prompts(pipe, [VALIDATION_NEGATIVE], DEVICE, WEIGHT_DTYPE)[0]
    latents = [A.encode_images(pipe, to_tensor(im), DEVICE, WEIGHT_DTYPE) for im in tqdm(processed, desc="VAE")]

offload(OTHER_MODULES, "cpu")
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

assert len(caption_conds) == len(df) == len(latents)
for i, (c, z) in enumerate(zip(caption_conds, latents)):
    assert torch.isfinite(z).all(), f"{df.file[i]} 잠재값에 NaN/Inf"
    for k, v in c.items():
        if torch.is_tensor(v) and v.is_floating_point():
            assert torch.isfinite(v).all(), f"{df.file[i]} 텍스트 임베딩 {k} 에 NaN/Inf"
print("잠재값 모양 예:", tuple(latents[0].shape), "| 텍스트 키:", sorted(caption_conds[0].keys()))

## 3. 시각화

In [ ]:
n = len(processed)
cols = min(6, n)
rows_ = math.ceil(n / cols)
fig, axes = plt.subplots(rows_, cols, figsize=(3 * cols, 3.6 * rows_), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, im, cap, b in zip(axes.flat, processed, df.train_caption, df.bucket):
    ax.imshow(im)
    ax.set_title(f"{b[0]}x{b[1]}\n" + (cap[:38] + "…" if len(cap) > 38 else cap), fontsize=7)
plt.suptitle("전처리된 학습 이미지와 캡션"); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
axes[0].hist(df.aspect, bins=12, color="#7B3342"); axes[0].set_title("원본 가로/세로 비율")
bc = Counter(f"{w}x{h}" for w, h in df.bucket)
axes[1].bar(list(bc.keys()), list(bc.values()), color="#4A372E"); axes[1].set_title("묶음별 장수")
axes[1].tick_params(axis="x", rotation=45)
axes[2].hist(df.caption_words, bins=10, color="#4A4A52"); axes[2].set_title("캡션 낱말 수")
z_mean = [float(z.mean()) for z in latents]; z_std = [float(z.std()) for z in latents]
axes[3].scatter(z_mean, z_std, color="#6B4A35"); axes[3].set_title("잠재값 평균-표준편차")
axes[3].set_xlabel("mean"); axes[3].set_ylabel("std")
plt.tight_layout(); plt.show()
print(f"잠재값 전체 평균 {np.mean(z_mean):.3f}, 표준편차 평균 {np.mean(z_std):.3f} (정규화가 맞으면 대략 0, 1 근처)")

## 4. 학습

- LoRA 를 디노이저의 어텐션 층(어댑터의 `LORA_TARGET_MODULES`)에 붙이고, 그 층만 학습한다.
- 노이즈 단계: flow 계열은 logit-normal(0, 1) 로 시그마를 뽑고, eps/v 계열은 균등하게 뽑는다.
- `save_every` 마다 LoRA 파일을, `validate_every` 마다 검증 손실과 고정 시드 샘플을 남긴다.
- 손실이 NaN/Inf 인 단계는 건너뛰고, `max_nan_steps` 를 넘으면 멈춘다.

In [ ]:
RUN_DIR = Path(USER["OUTPUT_DIR"]) / (USER["RUN_NAME"] + (f"_test_{TEST_FAMILY}" if TEST_FAMILY else ""))
if RUN_DIR.exists() and TEST_FAMILY:
    shutil.rmtree(RUN_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)

lora_cfg = LoraConfig(r=FIXED["rank"], lora_alpha=FIXED["alpha"], lora_dropout=FIXED["dropout"],
                      init_lora_weights=FIXED["init_lora_weights"], target_modules=list(A.LORA_TARGET_MODULES))
denoiser.to(DEVICE)
denoiser.add_adapter(lora_cfg)
lora_params = [p for n, p in denoiser.named_parameters() if "lora_" in n]
if not lora_params:
    raise RuntimeError(f"LoRA 를 붙일 층을 찾지 못했습니다: {A.LORA_TARGET_MODULES}")
for n, p in denoiser.named_parameters():
    p.requires_grad_("lora_" in n)
for p in lora_params:            # 학습 파라미터는 fp32 (공식 스크립트의 cast_training_params 와 같음)
    p.data = p.data.float()
n_targets = len({n.rsplit(".lora_", 1)[0] for n, _ in denoiser.named_parameters() if "lora_A" in n})
print(f"LoRA 층 {n_targets}개, 학습 파라미터 {sum(p.numel() for p in lora_params) / 1e6:.2f}M")

def to_dev(cond):
    return {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in cond.items()}

# diffusers 0.42.1 의 Chroma 는 체크포인트 경로에서 single block 에 attention_mask 를 넘기지 않는다
# (학습 때만 패딩 토큰을 보게 됨). 그래서 Chroma 는 끄고, 나머지는 켠 뒤 같은 입력의 손실이 같은지 확인한다.
NO_GC_FAMILIES = {"chroma"}
denoiser.train()
if FIXED["gradient_checkpointing"] and A.FAMILY not in NO_GC_FAMILIES \
        and getattr(denoiser, "_supports_gradient_checkpointing", False):
    _g = torch.Generator("cpu").manual_seed(0)
    _x0 = latents[0].to(DEVICE); _nz = torch.randn(_x0.shape, generator=_g).to(DEVICE)
    _s = torch.tensor([0.5], device=DEVICE)
    with torch.no_grad():      # 기준값: 체크포인트 없는 경로, 활성값을 저장하지 않아 메모리를 쓰지 않음
        _l_off = float(A.forward_loss(pipe, denoiser, _x0, to_dev(caption_conds[0]), _nz, _s))
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    denoiser.enable_gradient_checkpointing()
    with torch.enable_grad():  # 체크포인트 경로는 기울기를 계산할 때만 쓰인다
        _l_on = float(A.forward_loss(pipe, denoiser, _x0, to_dev(caption_conds[0]), _nz, _s))
    denoiser.zero_grad(set_to_none=True)
    _tol = 1e-4 if WEIGHT_DTYPE == torch.float32 else 2e-2
    if abs(_l_on - _l_off) > _tol * max(1.0, abs(_l_off)):
        denoiser.disable_gradient_checkpointing()
        print(f"경고: 그래디언트 체크포인트를 켜면 손실이 달라져({_l_off:.5f} → {_l_on:.5f}) 끕니다. 메모리를 더 씁니다.")
    else:
        print(f"그래디언트 체크포인트 켬 (확인 손실 {_l_off:.5f} / {_l_on:.5f})")
else:
    print("그래디언트 체크포인트 끔")

optimizer = torch.optim.AdamW(lora_params, lr=FIXED["learning_rate"], betas=FIXED["betas"],
                              weight_decay=FIXED["weight_decay"], eps=FIXED["eps"])
lr_scheduler = get_scheduler(FIXED["lr_scheduler"], optimizer=optimizer, num_warmup_steps=FIXED["warmup_steps"],
                             num_training_steps=FIXED["max_train_steps"])

def train_sigma_shift():
    """공식 학습 레시피의 시그마 이동(shift). SD3·Z-Image 는 스케줄러의 고정 shift,
    LongCat 은 학습 해상도의 mu 로 exp(mu), 나머지 flow 계열(FLUX·Qwen·Krea2·Chroma)은 이동 없음."""
    cfg = pipe.scheduler.config
    if A.FAMILY in ("sd3", "z_image") and not cfg.get("use_dynamic_shifting", False):
        return float(cfg.get("shift", 1.0))
    if A.FAMILY == "longcat_image" and cfg.get("use_dynamic_shifting", False):
        seq = (TARGET // 16) ** 2                       # VAE 8배 축소 × 2x2 묶기
        m = (cfg["max_shift"] - cfg["base_shift"]) / (cfg["max_image_seq_len"] - cfg["base_image_seq_len"])
        return float(math.exp(seq * m + cfg["base_shift"] - m * cfg["base_image_seq_len"]))
    return 1.0

SIGMA_SHIFT = train_sigma_shift() if A.PREDICTION == "flow" else 1.0
print(f"시그마 이동(shift): {SIGMA_SHIFT:.3f}")

def sample_sigma(gen):
    if A.PREDICTION == "flow" and FIXED["timestep_sampling"] == "logit_normal":
        u = torch.normal(FIXED["logit_mean"], FIXED["logit_std"], (FIXED["batch_size"],), generator=gen)
        s = torch.sigmoid(u)
        if SIGMA_SHIFT != 1.0:
            s = SIGMA_SHIFT * s / (1 + (SIGMA_SHIFT - 1) * s)
        return s.clamp(1e-3, 1 - 1e-3)
    return torch.rand((FIXED["batch_size"],), generator=gen).clamp(1e-3, 1 - 1e-3)

# 검증 손실: 고정 이미지 · 고정 노이즈 · 고정 시그마 → 단계끼리 비교 가능
_vg = torch.Generator("cpu").manual_seed(FIXED["seed"] + 7)
VAL_IDX = list(range(min(FIXED["val_loss_images"], len(latents))))
VAL_NOISE = {i: torch.randn(latents[i].shape, generator=_vg) for i in VAL_IDX}

@torch.no_grad()
def validation_loss():
    denoiser.eval()
    vals = []
    for i in VAL_IDX:
        x0 = latents[i].to(DEVICE)
        for s in FIXED["val_sigmas"]:
            vals.append(float(A.forward_loss(pipe, denoiser, x0, to_dev(caption_conds[i]),
                                             VAL_NOISE[i].to(DEVICE), torch.tensor([s], device=DEVICE))))
    denoiser.train()
    return float(np.mean(vals))

SAMPLE_SIZE = (2 * M, 2 * M) if TEST_FAMILY else pick_bucket(2, 3)   # 세로 전신 비율
SAMPLE_STEPS = 2 if TEST_FAMILY else A.DEFAULT_STEPS

GEN_MODULES = [m for m in OTHER_MODULES if "text_encoder" not in m]   # 생성에 필요한 VAE 등 (텍스트 인코더는 CPU 에 둠)

@torch.no_grad()
def generate_images(conds, seeds):
    """캐시한 임베딩으로 고정 시드 이미지 생성. VAE 를 잠깐 GPU 에 올렸다가 내린다."""
    was_training = denoiser.training
    denoiser.eval()
    offload(GEN_MODULES, DEVICE)
    try:
        out = [A.generate(pipe, to_dev(c), to_dev(neg_cond), SAMPLE_SIZE[1], SAMPLE_SIZE[0], SAMPLE_STEPS,
                          A.DEFAULT_GUIDANCE, torch.Generator("cpu").manual_seed(s)) for c, s in zip(conds, seeds)]
    finally:
        offload(GEN_MODULES, "cpu")
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        denoiser.train(was_training)
    return out

def make_samples(step):
    out = generate_images(val_conds, VALIDATION_SEEDS)
    for img, seed in zip(out, VALIDATION_SEEDS):
        img.save(RUN_DIR / f"sample_step{step:05d}_seed{seed}.png")
    return out

def save_checkpoint(step):
    d = RUN_DIR / f"checkpoint-{step:05d}"
    d.mkdir(exist_ok=True)
    sd = {k: v.detach().to("cpu", torch.float32).contiguous() for k, v in get_peft_model_state_dict(denoiser).items()}
    return A.save_lora(pipe, sd, str(d))

history, val_history, sample_history, ckpt_paths = [], [], {}, {}
val_history.append((0, validation_loss()))
sample_history[0] = make_samples(0)             # LoRA B=0 으로 시작하므로 0 단계 = 베이스 모델
print(f"0 단계 검증 손실 {val_history[-1][1]:.4f}")

gen = torch.Generator("cpu").manual_seed(FIXED["seed"])
order, nan_steps, t_start = [], 0, time.time()
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
pbar = tqdm(range(1, FIXED["max_train_steps"] + 1), desc="학습")
for step in pbar:
    optimizer.zero_grad(set_to_none=True)
    step_loss = 0.0
    for _ in range(FIXED["grad_accum_steps"]):
        if not order:                       # 한 에폭마다 섞기 (시드 고정)
            order = torch.randperm(len(latents), generator=gen).tolist()
        i = order.pop()
        x0 = latents[i].to(DEVICE)
        noise = torch.randn(x0.shape, generator=gen).to(DEVICE)
        sigma = sample_sigma(gen).to(DEVICE)
        loss = A.forward_loss(pipe, denoiser, x0, to_dev(caption_conds[i]), noise, sigma)
        if not torch.isfinite(loss):
            break
        (loss / FIXED["grad_accum_steps"]).backward()
        step_loss += float(loss) / FIXED["grad_accum_steps"]
    grad_norm = float("nan")
    if math.isfinite(step_loss) and torch.isfinite(loss):
        grad_norm = float(torch.nn.utils.clip_grad_norm_(lora_params, FIXED["max_grad_norm"]))
    if not math.isfinite(grad_norm):     # 손실 또는 기울기가 NaN/Inf → 이 단계는 가중치를 바꾸지 않음
        nan_steps += 1
        optimizer.zero_grad(set_to_none=True)
        if nan_steps > FIXED["max_nan_steps"]:
            raise RuntimeError(f"NaN/Inf 손실·기울기가 {nan_steps}번 나와 멈춥니다. 학습률·데이터를 확인하세요.")
    else:
        optimizer.step(); lr_scheduler.step()
        history.append(dict(step=step, loss=step_loss, lr=lr_scheduler.get_last_lr()[0], grad_norm=grad_norm,
                            sigma=float(sigma.mean()), sec=time.time() - t_start))
        pbar.set_postfix(loss=f"{step_loss:.4f}", lr=f"{history[-1]['lr']:.1e}")
    # 건너뛴 단계여도 저장·검증은 한다 (가중치는 직전 단계와 같음)
    if step % FIXED["save_every"] == 0 or step == FIXED["max_train_steps"]:
        ckpt_paths[step] = save_checkpoint(step)
    if step % FIXED["validate_every"] == 0 or step == FIXED["max_train_steps"]:
        val_history.append((step, validation_loss()))
        sample_history[step] = make_samples(step)

train_sec = time.time() - t_start
peak_gb = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else float("nan")
pd.DataFrame(history).to_csv(RUN_DIR / "train_log.csv", index=False)
print(f"학습 {len(history)}단계, {train_sec / 60:.1f}분, NaN 건너뜀 {nan_steps}번, 최대 GPU 메모리 {peak_gb:.1f}GB")

## 5. 검증

### 5-1. 손실 곡선

In [ ]:
h = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(16, 3.4))
axes[0].plot(h.step, h.loss, alpha=0.35, label="단계 손실")
axes[0].plot(h.step, h.loss.ewm(alpha=0.05).mean(), label="이동 평균"); axes[0].legend(); axes[0].set_title("학습 손실")
vs, vl = zip(*val_history)
axes[1].plot(vs, vl, marker="o", color="#7B3342"); axes[1].set_title("검증 손실 (고정 노이즈)")
axes[2].plot(h.step, h.lr); axes[2].set_title("학습률")
plt.tight_layout(); plt.show()
_bad = [s for s, v in val_history if not math.isfinite(v)]
if _bad:
    print("경고: 검증 손실이 NaN/Inf 인 단계:", _bad)
best_step = min((s for s, v in val_history if s in ckpt_paths and math.isfinite(v)),
                key=lambda s: dict(val_history)[s], default=max(ckpt_paths))
print("단계별 검증 손실:", {s: round(v, 4) for s, v in val_history})
print("검증 손실이 가장 낮은 저장 단계:", best_step, "(최종 선택은 샘플을 사람이 보고 정한다)")

### 5-2. 단계별 샘플 비교 (같은 프롬프트·같은 시드)

In [ ]:
steps_ = sorted(sample_history)
fig, axes = plt.subplots(len(steps_), len(VALIDATION_PROMPTS), figsize=(3 * len(VALIDATION_PROMPTS), 3.3 * len(steps_)), squeeze=False)
for r, s in enumerate(steps_):
    for c, img in enumerate(sample_history[s]):
        ax = axes[r][c]; ax.imshow(img); ax.axis("off")
        if r == 0:
            ax.set_title(VALIDATION_PROMPTS[c][len(USER["TRIGGER"]) + 2:][:34], fontsize=7)
        if c == 0:
            ax.text(-0.05, 0.5, f"step {s}", transform=ax.transAxes, rotation=90, va="center", ha="right", fontsize=9)
plt.suptitle("0 단계 = 베이스 모델, 아래로 갈수록 학습이 진행된 LoRA"); plt.tight_layout(); plt.show()

### 5-3. 저장한 LoRA 파일 다시 불러오기 확인

학습한 어댑터를 지우고 마지막 저장 파일을 `load_lora_weights` 로 다시 불러와, 값이 같은지와 이미지가 나오는지 확인한다.

In [ ]:
final_step = max(ckpt_paths)
final_path = ckpt_paths[final_step]
trained = {k: v.detach().float().cpu() for k, v in get_peft_model_state_dict(denoiser).items()}
denoiser.delete_adapters("default") if hasattr(denoiser, "delete_adapters") else None
pipe.load_lora_weights(final_path if Path(final_path).is_file() else str(Path(final_path).parent), adapter_name="reload")
reloaded = {k: v.detach().float().cpu() for k, v in get_peft_model_state_dict(denoiser, adapter_name="reload").items()}
assert len(reloaded) == len(trained), (len(reloaded), len(trained))
max_diff = max(float((a - b).abs().max()) for a, b in zip(sorted(trained.items()), sorted(reloaded.items()))
               for a, b in [(a[1], b[1])])
print(f"다시 불러온 LoRA 텐서 {len(reloaded)}개, 최대 차이 {max_diff:.2e}")
assert max_diff < (1e-2 if WEIGHT_DTYPE != torch.float32 else 1e-5), "저장 파일과 학습 값이 다릅니다"
check_img = generate_images(val_conds[:1], VALIDATION_SEEDS[:1])[0]
assert np.asarray(check_img).std() > 0, "생성 이미지가 한 색입니다"
check_img.save(RUN_DIR / "reload_check.png")
print("다시 불러온 LoRA 로 생성 성공:", check_img.size)

### 5-4. 결과 요약 저장

In [ ]:
dataset_hash = hashlib.sha1("".join(sorted(df.sha1 + df.train_caption)).encode()).hexdigest()
summary = dict(
    family=A.FAMILY, model_path=USER["MODEL_PATH"] if not TEST_FAMILY else "tiny-test", run_dir=str(RUN_DIR),
    trigger=USER["TRIGGER"], images=len(df), dataset_sha1=dataset_hash, fixed=dict(FIXED),
    steps_done=len(history), nan_steps=nan_steps, train_minutes=round(train_sec / 60, 2),
    peak_gpu_gb=None if math.isnan(peak_gb) else round(peak_gb, 2),
    final_train_loss=float(pd.DataFrame(history).loss.tail(20).mean()) if history else None,
    val_loss={int(s): round(v, 5) for s, v in val_history}, best_val_step=int(best_step),
    checkpoints={int(s): str(p) for s, p in ckpt_paths.items()},
    lora_file_mb=round(Path(final_path).stat().st_size / 1e6, 2) if Path(final_path).is_file() else None,
    versions={k: _md.version(k) for k in PINNED},
)
(RUN_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=1, default=str))
if TEST_FAMILY:
    print(f"\nNOTEBOOK TEST OK {TEST_FAMILY}")